# Yambda — 50M / 500M: кандидаты, ранжирование и следующий временной срез

Цель — рекомендовать **100 различных треков**, предсказывая следующие 100 событий лайка каждого пользователя. Сначала измеряем, где теряются будущие лайки: вне исторического каталога → вне списка кандидатов → вне top-100. Затем выбираем источники по дополнительному покрытию и сравниваем три функции потерь CatBoost на одинаковых данных.

Все данные скачиваются с закреплённой ревизии Hugging Face. Кеш создаётся в текущем окружении; Google Drive и старые результаты не нужны. Код алгоритмов находится в ячейках. Единственный файл из репозитория конкурса — список официальных пользователей.

**Порядок запуска:** выберите NVIDIA GPU в Colab, выполните настройки и установку. Если установка попросит перезапустить среду, сделайте это, установите `INSTALL_DEPENDENCIES=False` и выполните notebook с начала. Далее работает обычный Run all. Сначала запускаем 50M. Для 500M можно передать небольшой `promotion.json` из законченного 50M-эксперимента или заново провести весь поиск.

Временной бюджет останавливает запуск **между этапами**, а нейросети — также между minibatch. Он не прерывает выполняющийся вызов SANSA/CatBoost. Завершённые источники сохраняются. Остановка по бюджету не считается успешным завершением эксперимента.

In [ ]:
DATA_SIZE = "50m"  # @param ["50m", "500m"]
RUN_LABEL = "fresh_v10_run01"
DEVICE = "GPU"  # @param ["GPU", "CPU"]
INSTALL_DEPENDENCIES = True
THREADS = 4
SEED = 42

RUN_SANSA = True
RUN_NEURAL = True
RUN_OPTIONAL = DATA_SIZE == "50m"  # BPR and Mult-VAE broaden the 50M screen.
SESSION_COMPUTE_MINUTES = 90 if DATA_SIZE == "50m" else 120
NEURAL_MINUTES_PER_SOURCE = 6 if DATA_SIZE == "50m" else 10

# Optional: upload promotion.json from the 50M result ZIP, then set its exact path.
# Empty means a fresh broad source screen; no prior measurements are embedded.
PROMOTION_PATH = ""

RUN_SUBMISSION = False
ALLOW_PARTIAL_HISTORY_SUBMISSION = False
DATA_REVISION = "dd6f3a19eef5866e346c3270e098baa641a44948"
USERS_REPO_REVISION = "e3acd135b5d61a504eb6f4f9577b6dc9c07fd4b2"

## Окружение

GPU используется для PyTorch и CatBoost. Разреженные графы, SANSA и компактные ALS обучаются на CPU; наличие GPU не ускоряет автоматически эти вычисления. ALS получает несколько осмысленных вариантов confidence/regularization, а dot-product и cosine retrieval сравниваются без повторного обучения факторов.

Системные библиотеки SANSA устанавливаются автоматически только в Colab. В локальном окружении зависимости устанавливаются в активный Python; если в нём нет pip, используется uv. Установщик не меняет PyTorch: Colab предоставляет его вместе с CUDA.

In [ ]:
# Notebook setup cell: runs only when the user executes it.
import importlib.util
import os
from pathlib import Path
import shutil
import subprocess
import sys

IN_COLAB = (
    importlib.util.find_spec("google.colab") is not None
    if importlib.util.find_spec("google")
    else False
)
ROOT = Path("/content/yambda_v10") if IN_COLAB else Path.home() / "yambda_runs" / "v10"
ROOT.mkdir(parents=True, exist_ok=True)
os.environ["POLARS_MAX_THREADS"] = str(THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = str(THREADS)

if INSTALL_DEPENDENCIES:
    if RUN_SANSA:
        if IN_COLAB:
            subprocess.run(["apt-get", "update", "-qq"], check=True)
            subprocess.run(
                [
                    "apt-get",
                    "install",
                    "-y",
                    "-qq",
                    "build-essential",
                    "libsuitesparse-dev",
                    "pkg-config",
                ],
                check=True,
            )
        elif not Path("/usr/include/suitesparse/cholmod.h").exists():
            raise RuntimeError(
                "Install system headers in Bash: sudo apt-get update && sudo apt-get install -y build-essential libsuitesparse-dev pkg-config"
            )
    packages = [
        "numpy==2.3.5",
        "scipy==1.17.0",
        "pandas==2.3.3",
        "duckdb==1.4.3",
        "pyarrow==26.0.0",
        "implicit==0.7.3",
        "catboost==1.2.10",
        "huggingface-hub==1.33.0",
        "threadpoolctl==3.6.0",
        "matplotlib==3.10.8",
        "faiss-cpu==1.13.2",
        "numba==0.63.1",
        "nbformat>=5,<6",
        "ipython>=8,<10",
    ]
    if RUN_SANSA:
        packages += [
            "scikit-sparse==0.4.16",
            "sansa @ git+https://github.com/glami/sansa.git@735a782599cceb3663f8df14d75edfdf4c86874c",
        ]
    if importlib.util.find_spec("pip"):
        command = [sys.executable, "-m", "pip", "install", "--quiet"]
    elif shutil.which("uv"):
        command = ["uv", "pip", "install", "--python", sys.executable]
    else:
        raise RuntimeError(
            "No package installer. Use uv pip for the active environment."
        )
    # Installing over already-imported numerical packages requires one kernel restart.
    numerical_loaded = any(name in sys.modules for name in ["numpy", "scipy", "pandas"])
    subprocess.run(command + packages, check=True)
    if numerical_loaded:
        raise RuntimeError(
            "Dependencies installed. Restart the runtime/kernel, set INSTALL_DEPENDENCIES=False, and run from the first cell."
        )

import torch
from catboost.utils import get_gpu_device_count

if DEVICE == "GPU":
    if not torch.cuda.is_available() or not get_gpu_device_count():
        raise RuntimeError(
            "This profile requires an NVIDIA Colab GPU. Enable a GPU runtime; CPU mode must be selected explicitly."
        )
    TASK_TYPE, NEURAL_DEVICE = "GPU", "cuda"
else:
    TASK_TYPE = "CPU"
    NEURAL_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if RUN_NEURAL and NEURAL_DEVICE == "cpu":
        print(
            "Neural models will run on CPU with the per-model time cap; this is a weak-training pilot, not a fair GPU comparison."
        )
print("Root:", ROOT)
print("Python:", sys.version.split()[0], "PyTorch:", torch.__version__)
print("CatBoost:", TASK_TYPE, "Neural:", NEURAL_DEVICE)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Данные и временной протокол

`E` — конец доступных лайков. Три непересекающихся окна будущего имеют одинаковую длину 60 дней:

| Назначение | История строго раньше | Будущие метки |
|---|---|---|
| Обучение ranker | E−180 дней | [E−180, E−120) |
| Выбор источников, loss, деревьев и признаков | E−120 дней | [E−120, E−60) |
| Проверка зафиксированного решения | E−60 дней | [E−60, E) |

История внутри окна предсказания не обновляется. Когорта выбирается по наличию **прошлых** лайков до самого раннего среза, без отбора по будущей активности. Основная метрика — macro Recall@100 среди пользователей с 100 наблюдаемыми будущими событиями; параллельно выводятся active и all. Полная когорта смещена к активным пользователям, поэтому её результат нельзя автоматически переносить на всех пользователей скрытого теста. Если полных пользователей слишком мало, выбор модели останавливается.

По умолчанию сначала берутся следующие 100 событий, затем их item IDs превращаются в множество для знаменателя Recall. Повторный лайк допустим. `next100_new_items` — явно отдельный режим для проверки другой трактовки задания; он не смешивается с основным экспериментом. Порядок **внутри** уже выбранных 100 рекомендаций Recall@100 не меняет; важно, какие треки пересекли границу top-100.

In [ ]:
from __future__ import annotations

import contextlib
import dataclasses
import gc
import hashlib
import importlib.metadata
import json
import math
import resource
from pathlib import Path
import time
import zipfile

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy import sparse
from scipy.linalg import cho_factor, cho_solve
from threadpoolctl import threadpool_limits
from IPython.display import display

DAY = 86400
IMPLEMENTATION = "yambda-fresh-v10.0"


@dataclasses.dataclass
class Config:
    size: str = "50m"
    seed: int = 42
    threads: int = 4
    cohort: int = 2000
    history_days: int = 180
    listen_days: int = 90
    history_cap: int = 150
    catalog: int = 150000
    graph_items: int = 50000
    graph_neighbors: int = 60
    source_k: int = 300
    candidate_k: int = 1200
    max_sources: int = 6
    als_factors: int = 64
    als_iterations: int = 12
    sansa_items: int = 15000
    ease_items: int = 3000
    neural_items: int = 50000
    neural_epochs: int = 8
    neural_minutes: float = 8.0
    neural_examples: int = 100000
    neural_batch: int = 256
    negative_items: int = 256
    negative_rows: int = 250
    trees: tuple = (200, 400, 700)
    primary: str = "full100"
    minimum_complete: int = 50
    task_type: str = "GPU"
    neural_device: str = "cuda"
    run_neural: bool = True
    run_sansa: bool = True
    run_optional: bool = False
    target_mode: str = "next100_events"
    data_revision: str = "dd6f3a19eef5866e346c3270e098baa641a44948"


def fingerprint(value):
    return hashlib.sha256(
        json.dumps(value, sort_keys=True, default=str).encode()
    ).hexdigest()[:24]


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".partial")
    tmp.write_text(json.dumps(value, indent=2, default=str), encoding="utf-8")
    tmp.replace(path)


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for block in iter(lambda: file.read(8 * 1024**2), b""):
            digest.update(block)
    return digest.hexdigest()


def sql_path(path):
    return "'" + str(Path(path).resolve()).replace("'", "''") + "'"


class Experiment:
    """Owns bounded SQL, immutable configuration and small report artifacts."""

    def __init__(
        self, config, data, output, official_users=(), code_hash="interactive"
    ):
        self.cfg = config
        self.data, self.out = Path(data), Path(output)
        self.out.mkdir(parents=True, exist_ok=True)
        self.cache = self.out / "cache"
        self.cache.mkdir(exist_ok=True)
        self.official = np.asarray(official_users, dtype=np.int64)
        if len(np.unique(self.official)) != len(self.official):
            raise ValueError("Duplicate official UIDs")
        self.timings = []
        self.started = time.monotonic()
        self.budget_minutes = None
        self.versions = {
            name: importlib.metadata.version(name)
            for name in [
                "numpy",
                "scipy",
                "pandas",
                "duckdb",
                "pyarrow",
                "catboost",
                "implicit",
            ]
        }
        identity = {
            "implementation": IMPLEMENTATION,
            "code_hash": code_hash,
            "config": dataclasses.asdict(config),
            "versions": self.versions,
            "official": fingerprint(self.official.tolist()),
            "files": {
                p.name: [p.stat().st_size, p.stat().st_mtime_ns]
                for p in sorted(self.data.glob("*.parquet"))
            },
        }
        # Byte hashes from the download receipt bind raw inputs to the pinned revision.
        receipt = self.data / "download_receipt.json"
        if receipt.exists():
            identity["download"] = json.loads(receipt.read_text())
        for optional in ["torch", "faiss-cpu", "sansa"]:
            try:
                self.versions[optional] = importlib.metadata.version(optional)
            except importlib.metadata.PackageNotFoundError:
                pass
        self.identity = fingerprint(identity)
        manifest = self.out / "configuration.json"
        if manifest.exists() and fingerprint(
            json.loads(manifest.read_text())
        ) != fingerprint(identity):
            raise ValueError(
                "Configuration/inputs/code changed: choose a new RUN_LABEL"
            )
        write_json(manifest, identity)
        self.db = duckdb.connect()
        self.db.execute("CREATE MACRO exp2(x) AS pow(2.0,x)")
        self.db.execute("SET memory_limit='2GB'")
        self.db.execute(f"SET threads={config.threads}")
        self.db.execute("SET preserve_insertion_order=false")
        self.db.execute(f"SET temp_directory={sql_path(self.cache / 'spill')}")
        self.db.execute("SET max_temp_directory_size='20GB'")
        for name in ["likes", "listens", "dislikes", "unlikes", "undislikes"]:
            path = self.data / f"{name}.parquet"
            self.db.execute(
                f"CREATE VIEW {name} AS SELECT * FROM read_parquet({sql_path(path)})"
            )
        self.start, self.end = self.db.execute(
            "SELECT min(timestamp)::BIGINT, max(timestamp)::BIGINT+1 FROM likes"
        ).fetchone()
        self.cuts = {
            "train": self.end - 180 * DAY,
            "dev": self.end - 120 * DAY,
            "test": self.end - 60 * DAY,
        }
        if self.cuts["train"] < self.start + 30 * DAY:
            raise ValueError("Insufficient history for fixed chronological folds")
        self.cohort = self.choose_cohort()

    def check_budget(self):
        if (
            self.budget_minutes is not None
            and time.monotonic() - self.started > self.budget_minutes * 60
        ):
            write_report(self)
            raise RuntimeError(
                "Session compute budget reached. Completed sources are cached; resume the same notebook/run to continue. No result is labeled complete."
            )

    @contextlib.contextmanager
    def stage(self, name):
        self.check_budget()
        start = time.monotonic()
        print(
            f"BEGIN {name}; peakRSS={resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024**2:.2f} GiB",
            flush=True,
        )
        status = "ok"
        try:
            yield
        except BaseException:
            status = "failed"
            raise
        finally:
            seconds = time.monotonic() - start
            self.timings.append(
                dict(
                    stage=name,
                    seconds=seconds,
                    status=status,
                    peak_rss_gib=resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
                    / 1024**2,
                )
            )
            pd.DataFrame(self.timings).to_csv(self.out / "timings.csv", index=False)
            print(f"END {name}: {seconds / 60:.1f} min; {status}", flush=True)
            gc.collect()

    def table(self, name, frame):
        frame.to_csv(self.out / f"{name}.csv", index=False)
        display(frame)
        return frame

    def bind_plan(self, sources, promotion):
        plan = {"sources": list(sources), "promotion": promotion}
        path = self.out / "experiment_plan.json"
        if path.exists() and fingerprint(json.loads(path.read_text())) != fingerprint(
            plan
        ):
            raise ValueError("Source plan changed: use a new RUN_LABEL")
        write_json(path, plan)

    def choose_cohort(self):
        users = self.db.execute(
            f"SELECT DISTINCT uid::BIGINT AS uid FROM likes WHERE timestamp < {self.cuts['train']} ORDER BY uid"
        ).fetchnumpy()["uid"]
        rng = np.random.default_rng(self.cfg.seed)
        # Sample from the available population without looking at future activity.
        cohort = np.sort(
            rng.choice(users, min(len(users), self.cfg.cohort), replace=False)
        )
        pd.DataFrame(
            {"uid": cohort, "official": np.isin(cohort, self.official)}
        ).to_csv(self.out / "cohort.csv", index=False)
        return cohort

    def cached_sql(self, key, query):
        path = self.cache / f"{key}.parquet"
        if not path.exists():
            tmp = path.with_suffix(".partial.parquet")
            self.db.execute(
                f"COPY ({query}) TO {sql_path(tmp)} (FORMAT PARQUET, COMPRESSION ZSTD)"
            )
            tmp.replace(path)
        return path

    def targets(self, split, users=None):
        users = self.cohort if users is None else np.asarray(users)
        cut = self.cuts[split]
        stop = cut + 60 * DAY
        self.db.register("requested", pd.DataFrame({"uid": users}))
        condition = ""
        if self.cfg.target_mode == "next100_new_items":
            condition = f"AND NOT EXISTS (SELECT 1 FROM likes p WHERE p.uid=l.uid AND p.item_id=l.item_id AND p.timestamp<{cut})"
        query = f"""
            WITH future AS (
                SELECT l.uid::BIGINT AS uid, l.item_id::BIGINT AS item_id,
                       l.timestamp::BIGINT AS timestamp
                FROM likes l JOIN requested u USING(uid)
                WHERE l.timestamp >= {cut} AND l.timestamp < {stop} {condition}
            ), numbered AS (
                SELECT *, row_number() OVER (PARTITION BY uid ORDER BY timestamp, item_id) AS position
                FROM future
            ) SELECT * FROM numbered WHERE position<=100 ORDER BY uid, position
        """
        if self.cfg.target_mode == "next100_new_items":
            query = query.replace(
                "FROM future\n",
                "FROM (SELECT uid, item_id, min(timestamp) AS timestamp FROM future GROUP BY uid,item_id)\n",
            )
        frame = self.db.execute(query).df()
        truth = {int(u): set() for u in users}
        event_counts = dict.fromkeys(map(int, users), 0)
        for uid, group in frame.groupby("uid", sort=False):
            truth[int(uid)] = set(map(int, group.item_id))
            event_counts[int(uid)] = len(group)
        return truth, event_counts


def measure(bank, truth, counts, k=100, users=None):
    users = list(truth) if users is None else list(users)
    rows = []
    for uid in users:
        target = truth[uid]
        predicted = set(map(int, bank.get(uid, ([], []))[0][:k]))
        hits = len(target & predicted)
        rows.append(
            (
                uid,
                hits / len(target) if target else 0.0,
                bool(target),
                counts[uid] >= 100,
                hits,
                len(target),
            )
        )
    per_user = pd.DataFrame(
        rows, columns=["uid", "recall", "active", "full100", "hits", "target_size"]
    )
    result = {
        "all": per_user.recall.mean(),
        "active": per_user.loc[per_user.active, "recall"].mean(),
        "full100": per_user.loc[per_user.full100, "recall"].mean(),
        "n_users": len(per_user),
        "n_active": int(per_user.active.sum()),
        "n_full100": int(per_user.full100.sum()),
    }
    return result, per_user


def best_k(ids, values, k, positive=False):
    ids, values = np.asarray(ids, np.int64), np.asarray(values, np.float32)
    valid = np.isfinite(values)
    if not valid.all():
        raise ValueError("Non-finite retrieval scores")
    if positive:
        valid &= values > 0
    ids, values = ids[valid], values[valid]
    if len(ids) > k:
        keep = np.argpartition(values, -k)[-k:]
        ids, values = ids[keep], values[keep]
    order = np.lexsort((ids, -values))
    return ids[order], values[order]


def normalize_rows(matrix):
    lengths = np.sqrt(np.asarray(matrix.multiply(matrix).sum(axis=1)).ravel())
    return sparse.diags(1 / np.maximum(lengths, 1e-12)).dot(matrix).tocsr()


def row_topk(matrix, k, positive=True):
    matrix = matrix.tocsr()
    data, indices, indptr = [], [], [0]
    for row in range(matrix.shape[0]):
        a, b = matrix.indptr[row : row + 2]
        ids, scores = best_k(matrix.indices[a:b], matrix.data[a:b], k, positive)
        indices.extend(ids)
        data.extend(scores)
        indptr.append(len(indices))
    return sparse.csr_matrix(
        (
            np.asarray(data, np.float32),
            np.asarray(indices, np.int32),
            np.asarray(indptr, np.int64),
        ),
        shape=matrix.shape,
    )

In [ ]:
import urllib.request
from huggingface_hub import get_hf_file_metadata, hf_hub_download, hf_hub_url

DATA = ROOT / "data" / DATA_SIZE
DATA.mkdir(parents=True, exist_ok=True)
INPUTS = {
    **{
        f"{name}.parquet": f"flat/{DATA_SIZE}/{name}.parquet"
        for name in ["likes", "listens", "dislikes", "unlikes", "undislikes"]
    },
    "artist_item_mapping.parquet": "artist_item_mapping.parquet",
    "album_item_mapping.parquet": "album_item_mapping.parquet",
}
metadata = {}
for name, remote in INPUTS.items():
    info = get_hf_file_metadata(
        hf_hub_url("yandex/yambda", remote, repo_type="dataset", revision=DATA_REVISION)
    )
    metadata[name] = {"remote": remote, "size": info.size, "etag": info.etag.strip('"')}
remaining = sum(
    row["size"] for name, row in metadata.items() if not (DATA / name).exists()
)
free = shutil.disk_usage(ROOT).free
print(
    f"Download remaining: {remaining / 1024**3:.2f} GiB; free disk: {free / 1024**3:.1f} GiB"
)
if free < remaining + 20 * 1024**3:
    raise RuntimeError(
        "Less than 20 GiB working headroom after download; use a larger disk or a clean runtime."
    )
receipt_path = DATA / "download_receipt.json"
old_receipt = json.loads(receipt_path.read_text()) if receipt_path.exists() else {}
receipt = {"revision": DATA_REVISION, "files": {}}
for name, row in metadata.items():
    dest = DATA / name
    if not dest.exists():
        fetched = Path(
            hf_hub_download(
                "yandex/yambda",
                row["remote"],
                repo_type="dataset",
                revision=DATA_REVISION,
                cache_dir=ROOT / "hf_cache",
            )
        ).resolve()
        dest.symlink_to(fetched)
    stat = dest.stat()
    previous = old_receipt.get("files", {}).get(name, {})
    if (
        previous.get("size") == stat.st_size
        and previous.get("mtime_ns") == stat.st_mtime_ns
        and previous.get("sha256") == row["etag"]
    ):
        digest = previous["sha256"]
    else:
        digest = sha256_file(dest)
    if digest != row["etag"] or stat.st_size != row["size"]:
        raise ValueError("Pinned data checksum mismatch: " + name)
    receipt["files"][name] = {
        "sha256": digest,
        "size": stat.st_size,
        "mtime_ns": stat.st_mtime_ns,
    }
    print("Ready:", name)
write_json(receipt_path, receipt)

# This immutable repo revision supplies only the official UID list, never model code.
USERS = ROOT / "test_users.csv"
if not USERS.exists():
    url = f"https://raw.githubusercontent.com/DanTimson/yambda_contest/{USERS_REPO_REVISION}/input/test_users.csv"
    with urllib.request.urlopen(url) as response:
        USERS.write_bytes(response.read())
expected_users_hash = "c86803eb0c29fa1370596ef9171b4d622619fd113fef944d30ba91b2627e1f1e"
if sha256_file(USERS) != expected_users_hash:
    raise ValueError("Official user list differs from the pinned repository version")
official_users = pd.read_csv(USERS)["uid"].astype("int64").to_numpy()
if len(np.unique(official_users)) != len(official_users):
    raise ValueError("Duplicate official UIDs")
print("Official UIDs:", len(official_users), "SHA256:", sha256_file(USERS))

In [ ]:
CFG = Config(
    size=DATA_SIZE,
    seed=SEED,
    threads=THREADS,
    cohort=2000 if DATA_SIZE == "50m" else 3000,
    catalog=150000 if DATA_SIZE == "50m" else 250000,
    graph_items=40000 if DATA_SIZE == "50m" else 60000,
    sansa_items=15000,
    neural_items=50000 if DATA_SIZE == "50m" else 80000,
    neural_examples=60000 if DATA_SIZE == "50m" else 120000,
    neural_minutes=NEURAL_MINUTES_PER_SOURCE,
    source_k=300,
    candidate_k=1200,
    max_sources=6,
    task_type=TASK_TYPE,
    neural_device=NEURAL_DEVICE,
    run_neural=RUN_NEURAL,
    run_sansa=RUN_SANSA,
    run_optional=RUN_OPTIONAL,
)
ENGINE_SHA256 = "945d2e28f3fcf2ff098b3100c1c13d6309a59317331bf7d2c13abef1ec4d2556"
RUNS = ROOT / "runs" / (DATA_SIZE + "_" + RUN_LABEL)
E = Experiment(CFG, DATA, RUNS, official_users, code_hash=ENGINE_SHA256)
E.budget_minutes = SESSION_COMPUTE_MINUTES
print("Results:", RUNS)
print("Cutoffs:", E.cuts)

## Подготовка прошлого без загрузки сырых listens в RAM

DuckDB агрегирует события на диск с лимитом памяти 2 GiB и разрешённым spill. Для популяционной collaborative-модели берётся ограниченное число последних пар пользователя. **Для персональных признаков когорты** сохраняются все прошлые лайки и все пары прослушиваний внутри окна, поэтому ограничение популяционной матрицы не превращает давний лайк в ложный «новый трек».

Каталог каждой модели выбирается только по прошлому. Первое появление в логах не равно дате релиза; соответствующий признак означает только наблюдаемый возраст. Для треков с несколькими артистами/альбомами используется минимальный ID как воспроизводимый основной кредит. Это упрощение явно ограничивает metadata-признаки.

In [ ]:
class State:
    """Past-only snapshot. Disk SQL first; bounded population histories in RAM."""

    def __init__(self, experiment, cut, users):
        self.e, self.cfg, self.cut = experiment, experiment.cfg, int(cut)
        self.users = np.asarray(users, np.int64)
        self.folder = self.e.cache / f"cut_{cut}"
        self.folder.mkdir(exist_ok=True)
        self.banks, self.catalogs, self.source_seconds = {}, {}, {}
        self.factor_features = {}
        self.catalog_rows = []
        self._prepare()

    def read(self, name, query):
        path = self.e.cached_sql(f"{self.cut}_{name}", query)
        return pd.read_parquet(path)

    def _prepare(self):
        c, cfg = self.cut, self.cfg
        with self.e.stage(f"past aggregates {c}"):
            # Every arithmetic operation uses signed/double timestamps before subtraction.
            self.stats = (
                self.read(
                    "stats",
                    f"""
                SELECT item_id::BIGINT AS item_id, count(*)::DOUBLE AS pop,
                       count(*) FILTER(WHERE timestamp>={c - 30 * DAY})::DOUBLE AS pop30,
                       count(*) FILTER(WHERE timestamp>={c - 7 * DAY})::DOUBLE AS pop7,
                       count(*) FILTER(WHERE timestamp>={c - 2 * DAY})::DOUBLE AS pop2,
                       sum(exp2((timestamp::DOUBLE-{c})/{30 * DAY})) AS decay30,
                       sum(exp2((timestamp::DOUBLE-{c})/{7 * DAY})) AS decay7,
                       avg(is_organic::DOUBLE) AS organic_share,
                       min(timestamp)::BIGINT AS first_time
                FROM likes WHERE timestamp<{c} GROUP BY item_id
            """,
                )
                .set_index("item_id")
                .sort_index()
            )
            self.like = self.read(
                "like_history",
                f"""
                WITH pairs AS (
                    SELECT uid::BIGINT AS uid, item_id::BIGINT AS item_id,
                           max(timestamp)::BIGINT AS timestamp, count(*)::INTEGER AS n,
                           avg(is_organic::DOUBLE) AS organic
                    FROM likes WHERE timestamp < {c} AND timestamp >= {c - cfg.history_days * DAY}
                    GROUP BY uid,item_id
                ) SELECT * FROM pairs
                QUALIFY row_number() OVER(PARTITION BY uid ORDER BY timestamp DESC,item_id)<={cfg.history_cap}
                ORDER BY uid,timestamp,item_id
            """,
            )
            listen_aggregate = self.e.cached_sql(
                f"{c}_listen_aggregate",
                f"""
                SELECT uid::BIGINT AS uid, item_id::BIGINT AS item_id,
                       max(timestamp)::BIGINT AS timestamp, count(*)::INTEGER AS n,
                       count(*) FILTER(WHERE played_ratio_pct>=50 AND track_length_seconds>0)::INTEGER AS qualified,
                       count(*) FILTER(WHERE played_ratio_pct>=100 AND track_length_seconds>0)::INTEGER AS full,
                       count(*) FILTER(WHERE played_ratio_pct<20)::INTEGER AS short,
                       avg(is_organic::DOUBLE) AS organic,
                       count(*) FILTER(WHERE timestamp>={c - 7 * DAY})::INTEGER AS recent7
                FROM listens WHERE timestamp < {c} AND timestamp >= {c - cfg.listen_days * DAY}
                GROUP BY uid,item_id
            """,
            )
            self.listen = self.read(
                "listen_history",
                f"""
                SELECT * FROM read_parquet({sql_path(listen_aggregate)})
                QUALIFY row_number() OVER(PARTITION BY uid ORDER BY timestamp DESC,item_id)<={cfg.history_cap}
                ORDER BY uid,timestamp,item_id
            """,
            )
            # Catalog selection depends only on historical popularity, not future targets.
            counts = self.listen.groupby("item_id").qualified.sum()
            candidate_pop = self.stats.decay30.add(counts * 0.05, fill_value=0)
            self.items = np.sort(
                candidate_pop.nlargest(cfg.catalog).index.to_numpy(np.int64)
            )
            self.population = np.union1d(
                self.like.uid.unique(), self.listen.uid.unique()
            )
            # One fixed empty row handles every unknown UID without adding thousands
            # of official-only users to the population used by BM25 and UserKNN.
            self.population = np.r_[self.population, np.iinfo(np.int64).max]
            rows = np.searchsorted(self.population, self.users)
            known = self.population[rows] == self.users
            self.user_rows = np.where(known, rows, len(self.population) - 1)
            self.query_rows = dict(zip(map(int, self.users), map(int, self.user_rows)))
            self.L = self.matrix(self.like, "like")
            self.Q = self.matrix(self.listen[self.listen.qualified > 0], "listen")
            self.B = self.L.copy()
            self.B.data[:] = 1
            self.catalog_rows.append(
                dict(
                    source="shared",
                    catalog=len(self.items),
                    population=len(self.population),
                    like_pairs=self.L.nnz,
                    listen_pairs=self.Q.nnz,
                )
            )
            self.e.db.register("selected_items", pd.DataFrame({"item_id": self.items}))
            for kind in ["artist", "album"]:
                path = self.e.data / f"{kind}_item_mapping.parquet"
                mapping = self.e.db.execute(f"""
                    SELECT m.item_id::BIGINT AS item_id, min(m.{kind}_id)::BIGINT AS {kind}_id
                    FROM read_parquet({sql_path(path)}) m JOIN selected_items i USING(item_id)
                    GROUP BY m.item_id
                """).df()
                setattr(self, kind, mapping.set_index("item_id")[f"{kind}_id"])
            self.e.db.register("selected_users", pd.DataFrame({"uid": self.users}))
            full_likes = self.e.db.execute(f"""
                SELECT l.uid::BIGINT AS uid,l.item_id::BIGINT AS item_id,
                       max(timestamp)::BIGINT AS timestamp,count(*)::INTEGER AS n,
                       avg(is_organic::DOUBLE) AS organic
                FROM likes l JOIN selected_users u USING(uid) WHERE timestamp<{c}
                GROUP BY l.uid,l.item_id ORDER BY l.uid,timestamp,l.item_id
            """).df()
            full_listens = self.e.db.execute(f"""
                SELECT p.* FROM read_parquet({sql_path(listen_aggregate)}) p
                JOIN selected_users u USING(uid) ORDER BY uid,timestamp,item_id
            """).df()
            self.like_by_user = {
                int(u): g for u, g in full_likes.groupby("uid", sort=False)
            }
            self.listen_by_user = {
                int(u): g for u, g in full_listens.groupby("uid", sort=False)
            }
            self.feedback = self.e.db.execute(f"""
                WITH actions AS (
                    SELECT uid,item_id,timestamp,1 AS dislike,0 AS unlike FROM dislikes WHERE timestamp<{c}
                    UNION ALL SELECT uid,item_id,timestamp,0,0 FROM undislikes WHERE timestamp<{c}
                    UNION ALL SELECT uid,item_id,timestamp,0,1 FROM unlikes WHERE timestamp<{c}
                ) SELECT uid::BIGINT AS uid,item_id::BIGINT AS item_id,
                         max(timestamp)::BIGINT AS timestamp,
                         max(dislike)::INTEGER AS ever_disliked,max(unlike)::INTEGER AS ever_unliked
                  FROM actions JOIN selected_users USING(uid) GROUP BY uid,item_id
            """).df()
            self.feedback_by_user = {
                int(u): g.set_index("item_id") for u, g in self.feedback.groupby("uid")
            }

    def matrix(self, frame, kind):
        frame = frame[frame.item_id.isin(self.items)]
        rows = np.searchsorted(self.population, frame.uid.to_numpy())
        cols = np.searchsorted(self.items, frame.item_id.to_numpy())
        age = (float(self.cut) - frame.timestamp.to_numpy(dtype=float)) / DAY
        if kind == "like":
            weights = 1 + 4 * np.exp2(-age / 60)
        else:
            weights = (
                1 + 2 * np.log1p(np.minimum(frame.qualified.to_numpy(), 20))
            ) * np.exp2(-age / 30)
        return sparse.csr_matrix(
            (weights.astype(np.float32), (rows, cols)),
            shape=(len(self.population), len(self.items)),
        )

    def store_bank(self, name, bank, catalog, seconds=0):
        normalized = {}
        for uid in self.users:
            ids, scores = bank.get(int(uid), ([], []))
            ids, scores = best_k(ids, scores, self.cfg.source_k)
            if len(np.unique(ids)) != len(ids):
                raise ValueError(f"Duplicate candidates: {name}/{uid}")
            normalized[int(uid)] = (ids, scores)
        self.banks[name] = normalized
        self.catalogs[name] = np.asarray(catalog, np.int64)
        self.source_seconds[name] = float(seconds)
        flat = [
            (u, int(i), float(s), rank)
            for u, (ids, scores) in normalized.items()
            for rank, (i, s) in enumerate(zip(ids, scores))
        ]
        path = self.bank_path(name)
        tmp = path.with_suffix(".partial.parquet")
        pd.DataFrame(flat, columns=["uid", "item_id", "score", "rank"]).to_parquet(
            tmp, index=False
        )
        tmp.replace(path)
        np.save(path.with_suffix(".catalog.npy"), np.asarray(catalog, np.int64))
        write_json(
            path.with_suffix(".json"),
            {"seconds": seconds, "users": fingerprint(self.users.tolist())},
        )

    def bank_path(self, name):
        return self.folder / f"bank_{name}_{fingerprint(self.users.tolist())}.parquet"

    def cached_bank(self, name):
        path = self.bank_path(name)
        receipt = path.with_suffix(".json")
        if not path.exists() or not receipt.exists():
            return False
        frame = pd.read_parquet(path).sort_values(["uid", "rank"])
        bank = {
            int(u): (g.item_id.to_numpy(np.int64), g.score.to_numpy(np.float32))
            for u, g in frame.groupby("uid")
        }
        self.banks[name] = {
            int(u): bank.get(int(u), (np.array([], np.int64), np.array([], np.float32)))
            for u in self.users
        }
        self.catalogs[name] = np.load(path.with_suffix(".catalog.npy"))
        self.source_seconds[name] = json.loads(receipt.read_text())["seconds"]
        return True

    def sparse_bank(self, scores, columns=None):
        items = self.items if columns is None else self.items[columns]
        bank = {}
        scores = scores.tocsr()
        for row, uid in enumerate(self.users):
            a, b = scores.indptr[row : row + 2]
            bank[int(uid)] = best_k(
                items[scores.indices[a:b]], scores.data[a:b], self.cfg.source_k, True
            )
        return bank

    def dense_bank(self, user_vectors, item_vectors, items, cosine=False):
        bank = {}
        if cosine:
            user_vectors = user_vectors / np.maximum(
                np.linalg.norm(user_vectors, axis=1, keepdims=True), 1e-12
            )
            item_vectors = item_vectors / np.maximum(
                np.linalg.norm(item_vectors, axis=1, keepdims=True), 1e-12
            )
        # Bounded exact retrieval; no sampled evaluation catalog or ANN recall confound.
        for start in range(0, len(self.users), 32):
            scores = user_vectors[start : start + 32] @ item_vectors.T
            for row, values in enumerate(scores, start):
                bank[int(self.users[row])] = (
                    best_k(items, values, self.cfg.source_k)
                    if np.linalg.norm(user_vectors[row]) > 1e-12
                    else ([], [])
                )
        return bank

## Источники кандидатов

| Семейство | Проверяемая гипотеза |
|---|---|
| DecayPop 7/30 дней | Различная скорость музыкальных трендов |
| Любимые артисты и альбомы | Персональный вкус + свежая популярность внутри группы |
| Heard-unliked | Отложенные лайки знакомых по прослушиваниям треков |
| ALS: обычный, сильный confidence, BM25 input, mixed actions | Влияние веса наблюдений и совместного пространства лайков/прослушиваний |
| ALS dot / cosine | Потери из-за геометрии retrieval, независимо от факторов |
| Item cosine / UserKNN | Соседи треков и свежие лайки похожих пользователей |
| Like→like / listen→like | Направленные переходы в пределах 7 дней с decay и поправкой на популярность |
| EASE / SANSA | Линейные реконструкторы пользовательской истории |
| Two-tower / SASRec | Смешанная история + metadata; порядок недавних лайков |
| Mult-VAE / BPR | Дополнительные семейства для широкого 50M-поиска |

Для направленных переходов используются не более 30 последних исходных и 30 целевых событий на пользователя: это ограничивает join и делает стоимость явной. Прослушивание без лайка — слабый сигнал, не доказанный dislike. Негативы для supervised-обучения означают отсутствие целевого лайка в наблюдаемом окне, а не причинно установленный отказ.

Кандидаты уже лайкнутых треков не запрещаются автоматически. Не используем будущие релизы, взаимодействия или признаки для исторических срезов.

In [ ]:
def simple_sources(state, requested):
    cfg, c = state.cfg, state.cut
    global_items = state.stats.index.to_numpy(np.int64)
    for name, column in [("decay7", "decay7"), ("decay30", "decay30")]:
        if name not in requested or state.cached_bank(name):
            continue
        ids, scores = best_k(global_items, state.stats[column].to_numpy(), cfg.source_k)
        state.store_bank(
            name, {int(u): (ids, scores) for u in state.users}, global_items
        )
    for kind in ["artist", "album"]:
        name = f"favorite_{kind}"
        if name not in requested or state.cached_bank(name):
            continue
        start = time.monotonic()
        mapping = getattr(state, kind)
        metadata = pd.DataFrame(
            {
                "item_id": state.items,
                "group": mapping.reindex(state.items).fillna(-1).to_numpy(),
            }
        )
        metadata["pop"] = state.stats.decay30.reindex(state.items).fillna(0).to_numpy()
        by_group = {
            int(g): part.nlargest(100, "pop")
            for g, part in metadata[metadata.group >= 0].groupby("group")
        }
        bank = {}
        for uid in state.users:
            history = state.like_by_user.get(int(uid))
            if history is None:
                bank[int(uid)] = ([], [])
                continue
            groups = mapping.reindex(history.item_id).fillna(-1).to_numpy()
            weights = np.exp2(
                (history.timestamp.to_numpy(dtype=float) - c) / (60 * DAY)
            )
            tastes = pd.Series(weights).groupby(groups).sum().nlargest(5)
            scores = {}
            for group, weight in tastes.items():
                for row in by_group.get(
                    int(group), pd.DataFrame(columns=["item_id", "pop"])
                ).itertuples():
                    scores[int(row.item_id)] = float(weight) * math.log1p(row.pop)
            bank[int(uid)] = best_k(list(scores), list(scores.values()), cfg.source_k)
        state.store_bank(name, bank, state.items, time.monotonic() - start)
    if "heard_unliked" in requested and not state.cached_bank("heard_unliked"):
        bank = {}
        for uid in state.users:
            heard = state.listen_by_user.get(int(uid))
            liked = state.like_by_user.get(int(uid))
            if heard is None:
                bank[int(uid)] = ([], [])
                continue
            score = np.log1p(heard.qualified.to_numpy()) * np.exp2(
                (heard.timestamp.to_numpy(dtype=float) - c) / (30 * DAY)
            )
            if liked is not None:
                score *= np.where(heard.item_id.isin(liked.item_id), 0.2, 1.0)
            bank[int(uid)] = best_k(heard.item_id, score, cfg.source_k, True)
        heard_catalog = (
            np.unique(
                np.concatenate(
                    [
                        g.item_id.to_numpy(np.int64)
                        for g in state.listen_by_user.values()
                    ]
                )
            )
            if state.listen_by_user
            else np.array([], np.int64)
        )
        state.store_bank("heard_unliked", bank, heard_catalog)

In [ ]:
def als_sources(state, requested):
    from implicit.als import AlternatingLeastSquares
    from implicit.nearest_neighbours import bm25_weight

    recipes = {
        "als_likes": ("likes", 1.0, 0.1, False),
        "als_strong": ("likes", 20.0, 1.0, False),
        "als_bm25": ("binary", 10.0, 1.0, True),
        "als_mixed": ("mixed", 5.0, 0.5, False),
    }
    for name, (matrix_kind, alpha, regularization, use_bm25) in recipes.items():
        needed = [
            n
            for n in [name, name + "_cos"]
            if n in requested and not state.cached_bank(n)
        ]
        if not needed:
            continue
        start = time.monotonic()
        model = AlternatingLeastSquares(
            factors=state.cfg.als_factors,
            iterations=state.cfg.als_iterations,
            regularization=regularization,
            alpha=alpha,
            random_state=state.cfg.seed,
            use_gpu=False,
            num_threads=state.cfg.threads,
        )
        model_path = state.folder / f"{name}_model.npz"
        if model_path.exists():
            saved = np.load(model_path)
            uf, vf = saved["users"], saved["items"]
        else:
            matrix = (
                (state.L + state.Q * 0.3).tocsr()
                if matrix_kind == "mixed"
                else (state.B if matrix_kind == "binary" else state.L)
            )
            train = bm25_weight(matrix, K1=100, B=0.8).tocsr() if use_bm25 else matrix
            # Confidence must remain nonnegative (BM25 IDF can be negative).
            train = train.copy()
            train.data = np.maximum(train.data, 0)
            train.eliminate_zeros()
            with threadpool_limits(limits=1, user_api="blas"):
                model.fit(train, show_progress=False)
            uf, vf = model.user_factors, model.item_factors
            tmp = model_path.with_suffix(".partial.npz")
            np.savez(tmp, users=uf, items=vf)
            tmp.replace(model_path)
        for target in needed:
            bank = state.dense_bank(
                uf[state.user_rows], vf, state.items, target.endswith("_cos")
            )
            state.store_bank(target, bank, state.items, time.monotonic() - start)
        del model, uf, vf
        gc.collect()

In [ ]:
def neighborhood_sources(state, requested):
    needed = [
        n
        for n in ["item_cosine", "user_knn"]
        if n in requested and not state.cached_bank(n)
    ]
    if not needed:
        return
    cfg = state.cfg
    count = np.asarray(state.B.sum(axis=0)).ravel()
    cols = np.sort(
        np.argsort(-count, kind="stable")[: min(cfg.graph_items, len(count))]
    )
    items = state.items[cols]
    X = state.B[:, cols].tocsr()
    for name in ["item_cosine", "user_knn"]:
        if name not in requested or state.cached_bank(name):
            continue
        start = time.monotonic()
        if name == "item_cosine":
            path = state.folder / "item_cosine.npz"
            if path.exists():
                graph = sparse.load_npz(path)
            else:
                normalized = normalize_rows(X.T).tocsr()
                blocks = []
                for left in range(0, len(cols), 64):
                    state.e.check_budget()
                    block = (normalized[left : left + 64] @ normalized.T).tocsr()
                    for row in range(block.shape[0]):
                        a, b = block.indptr[row : row + 2]
                        block.data[a:b][block.indices[a:b] == left + row] = 0
                    blocks.append(row_topk(block, cfg.graph_neighbors))
                graph = sparse.vstack(blocks, format="csr")
                sparse.save_npz(path, graph)
            scores = state.L[state.user_rows][:, cols] @ graph
        else:
            idf = np.log1p(len(state.population) / (1 + count[cols]))
            weighted = normalize_rows(X.multiply(idf)).tocsr()
            target = weighted[state.user_rows]
            blocks = []
            for left in range(0, len(state.users), 32):
                sim = (target[left : left + 32] @ weighted.T).tocsr()
                for row in range(sim.shape[0]):
                    a, b = sim.indptr[row : row + 2]
                    sim.data[a:b][sim.indices[a:b] == state.user_rows[left + row]] = 0
                blocks.append(row_topk(sim, 50))
            neighbors = sparse.vstack(blocks, format="csr")
            # Neighbors vote with recent likes, not just their most popular tracks.
            scores = neighbors @ state.L[:, cols]
        state.store_bank(
            name, state.sparse_bank(scores, cols), items, time.monotonic() - start
        )

In [ ]:
def transition_sources(state, requested):
    """Directed, capped temporal pairs from strictly historical raw events."""
    for name, origin in [("like_to_like", "likes"), ("listen_to_like", "listens")]:
        if name not in requested or state.cached_bank(name):
            continue
        start = time.monotonic()
        c, cfg = state.cut, state.cfg
        path = state.folder / f"{name}.npz"
        if path.exists():
            graph = sparse.load_npz(path)
        else:
            qualifier = (
                "AND played_ratio_pct>=50 AND track_length_seconds>0"
                if origin == "listens"
                else ""
            )
            # 30 latest origins and 30 latest destinations per user cap pair explosion.
            pairs = state.read(
                name + "_pairs",
                f"""
                WITH origins AS (
                    SELECT uid,item_id,timestamp FROM {origin}
                    WHERE timestamp>={c - 90 * DAY} AND timestamp<{c} {qualifier}
                    QUALIFY row_number() OVER(PARTITION BY uid ORDER BY timestamp DESC,item_id)<=30
                ), destinations AS (
                    SELECT uid,item_id,timestamp FROM likes
                    WHERE timestamp>={c - 90 * DAY} AND timestamp<{c}
                    QUALIFY row_number() OVER(PARTITION BY uid ORDER BY timestamp DESC,item_id)<=30
                ), per_user AS (
                    SELECT o.uid,o.item_id AS src,d.item_id AS dst,
                           max(exp2(-(d.timestamp::DOUBLE-o.timestamp::DOUBLE)/{3 * DAY}) *
                               exp2((d.timestamp::DOUBLE-{c})/{30 * DAY})) AS weight
                    FROM origins o JOIN destinations d ON o.uid=d.uid
                         AND d.timestamp>o.timestamp AND d.timestamp<=o.timestamp+{7 * DAY}
                         AND o.item_id<>d.item_id
                         AND o.item_id IN (SELECT item_id FROM selected_items)
                         AND d.item_id IN (SELECT item_id FROM selected_items)
                    GROUP BY o.uid,o.item_id,d.item_id
                ) SELECT src::BIGINT AS src,dst::BIGINT AS dst,sum(weight) AS weight,count(*) AS support
                  FROM per_user GROUP BY src,dst HAVING count(*)>=2
            """,
            )
            pairs = pairs[pairs.src.isin(state.items) & pairs.dst.isin(state.items)]
            row = np.searchsorted(state.items, pairs.src.to_numpy())
            col = np.searchsorted(state.items, pairs.dst.to_numpy())
            pop = state.stats["pop"].reindex(state.items).fillna(0).to_numpy()
            weights = pairs.weight.to_numpy() / np.power(1 + pop[col], 0.35)
            graph = sparse.csr_matrix(
                (weights.astype(np.float32), (row, col)),
                shape=(len(state.items), len(state.items)),
            )
            graph = row_topk(graph, cfg.graph_neighbors)
            sparse.save_npz(path, graph)
        profiles = state.L if origin == "likes" else state.Q
        scores = profiles[state.user_rows] @ graph
        state.store_bank(
            name, state.sparse_bank(scores), state.items, time.monotonic() - start
        )

In [ ]:
def linear_sources(state, requested):
    count = np.asarray(state.B.sum(axis=0)).ravel()
    for name, cap in [("ease", state.cfg.ease_items), ("sansa", state.cfg.sansa_items)]:
        if name not in requested or state.cached_bank(name):
            continue
        start = time.monotonic()
        cols = np.sort(np.argsort(-count, kind="stable")[: min(cap, len(count))])
        matrix = state.B[:, cols].tocsr()
        bank = {}
        if name == "ease":
            path = state.folder / "ease.npy"
            if path.exists():
                weights = np.load(path)
            else:
                gram = (matrix.T @ matrix).toarray().astype(np.float64)
                gram.flat[:: len(cols) + 1] += 500.0
                inverse = cho_solve(cho_factor(gram, lower=True), np.eye(len(cols)))
                weights = (-inverse / np.diag(inverse)[None, :]).astype(np.float32)
                np.fill_diagonal(weights, 0)
                np.save(path, weights)
            for left in range(0, len(state.users), 32):
                values = matrix[state.user_rows[left : left + 32]] @ weights
                for row, value in enumerate(values, left):
                    bank[int(state.users[row])] = best_k(
                        state.items[cols], value, state.cfg.source_k
                    )
        else:
            from sansa import (
                SANSA,
                SANSAConfig,
                ICFGramianFactorizerConfig,
                UMRUnitLowerTriangleInverterConfig,
            )

            model = SANSA(
                SANSAConfig(
                    l2=500,
                    weight_matrix_density=0.001,
                    gramian_factorizer_config=ICFGramianFactorizerConfig(),
                    lower_triangle_inverter_config=UMRUnitLowerTriangleInverterConfig(
                        scans=1, finetune_steps=5
                    ),
                )
            )
            paths = [
                state.folder / f"sansa_{part}.npz" for part in ["encoder", "decoder"]
            ]
            sansa_done = state.folder / "sansa_complete.json"
            if all(p.exists() for p in paths) and sansa_done.exists():
                model.load_weights(tuple(sparse.load_npz(p) for p in paths))
            else:
                with threadpool_limits(limits=state.cfg.threads, user_api="blas"):
                    model.fit(matrix)
                for p, w in zip(paths, model.weights):
                    temporary = p.with_suffix(".partial.npz")
                    sparse.save_npz(temporary, w)
                    temporary.replace(p)
                write_json(sansa_done, {"items": len(cols)})
            for left in range(0, len(state.users), 16):
                batch = matrix[state.user_rows[left : left + 16]]
                values = (model.forward(batch) + batch).toarray()
                for row, value in enumerate(values, left):
                    bank[int(state.users[row])] = best_k(
                        state.items[cols], value, state.cfg.source_k
                    )
        for row, uid in enumerate(state.users):
            if matrix[state.user_rows[row]].nnz == 0:
                bank[int(uid)] = (np.array([], np.int64), np.array([], np.float32))
        state.store_bank(name, bank, state.items[cols], time.monotonic() - start)

### Нейронные retrievers

Two-tower строит пользовательский вектор из предыдущих лайков и квалифицированных прослушиваний, с embedding типа события и основного артиста. Целевая песня не попадает в префикс через будущие прослушивания. SASRec использует causal attention над последними 50 лайками. Обе модели обучаются с in-batch и 256 общими случайными негативами; дубликаты положительного item маскируются.

Эпоха выбирается по внутренней проверке последнего прошлого события каждого пользователя. Это вспомогательный leave-last proxy внутри разрешённой истории, **не** внешний временной тест. Истории других пользователей внутри этого прошлого не обрезаются индивидуально под proxy. Внешняя оценка всегда использует глобальный срез. Если лимит времени остановил обучение рано, learning.csv показывает фактически выполненные шаги — слабый результат такого запуска не отвергает семейство модели.

На каталоге 50–80 тысяч треков сначала используем точный top-K по embedding similarity. Для two-tower дополнительно измеряется потеря HNSW относительно точного поиска. В CatBoost передаются scalar similarities и нормы, а не сырые координаты. Аудиоэмбеддинги из общего многогигабайтного файла в этот ограниченный по времени эксперимент не включены; two-tower использует ID и artist metadata, поэтому это не полноценная проверка cold-item audio retrieval.

In [ ]:
def neural_sources(state, requested):
    import torch
    from torch import nn
    from torch.nn import functional as F

    cfg = state.cfg
    names = [
        n
        for n in ["two_tower", "sasrec", "multvae"]
        if n in requested and not state.cached_bank(n)
    ]
    if not names:
        return
    device = torch.device(cfg.neural_device)
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError(
            "Neural CUDA/ROCm requested but unavailable. Select CPU explicitly or disable neural sources."
        )
    torch.set_num_threads(cfg.threads)
    count = np.asarray(state.B.sum(axis=0)).ravel()
    cols = np.sort(
        np.argsort(-count, kind="stable")[: min(cfg.neural_items, len(count))]
    )
    items = state.items[cols]
    n = len(items)
    raw = state.read(
        "neural_events",
        f"""
        SELECT uid::BIGINT AS uid,item_id::BIGINT AS item_id,timestamp::BIGINT AS timestamp
        FROM likes WHERE timestamp<{state.cut} AND timestamp>={state.cut - cfg.history_days * DAY}
        QUALIFY row_number() OVER(PARTITION BY uid ORDER BY timestamp DESC,item_id)<=65
        ORDER BY uid,timestamp,item_id
    """,
    )
    raw = raw[raw.item_id.isin(items)]
    sequences = {
        int(u): (
            np.searchsorted(items, g.item_id.to_numpy()) + 1,
            g.timestamp.to_numpy(),
        )
        for u, g in raw.groupby("uid", sort=False)
    }
    listened = state.listen[
        (state.listen.qualified > 0) & state.listen.item_id.isin(items)
    ]
    heard = {
        int(u): (
            np.searchsorted(items, g.item_id.to_numpy()) + 1,
            g.timestamp.to_numpy(),
        )
        for u, g in listened.groupby("uid", sort=False)
    }
    artist = state.artist.reindex(items).fillna(-1).to_numpy(np.int64)
    _, artist_tokens = np.unique(artist, return_inverse=True)
    artist_tokens = np.r_[0, artist_tokens + 1]
    seq_len = 50

    def prefixes(uid, tokens, times, position, mixed):
        t, actions, clocks = (
            tokens[:position],
            np.ones(position, np.int64),
            times[:position],
        )
        if mixed and uid in heard:
            heard_t, heard_time = heard[uid]
            before = heard_time < (
                times[position] if position < len(times) else state.cut
            )
            t = np.r_[t, heard_t[before]]
            actions = np.r_[actions, np.full(before.sum(), 2, np.int64)]
            clocks = np.r_[clocks, heard_time[before]]
        order = np.argsort(clocks, kind="stable")[-seq_len:]
        t, actions = t[order], actions[order]
        out = np.zeros((2, seq_len), np.int64)
        out[0, : len(t)] = t
        out[1, : len(t)] = actions
        return out

    class Retriever(nn.Module):
        def __init__(self, kind):
            super().__init__()
            self.kind = kind
            self.input = nn.Embedding(n + 1, 64, padding_idx=0)
            self.output = nn.Embedding(n + 1, 64, padding_idx=0)
            self.artist = nn.Embedding(int(artist_tokens.max()) + 1, 16, padding_idx=0)
            self.register_buffer("artists", torch.as_tensor(artist_tokens))
            self.action = nn.Embedding(3, 16, padding_idx=0)
            self.item_mlp = nn.Sequential(
                nn.Linear(80, 96), nn.GELU(), nn.Linear(96, 64)
            )
            self.user_mlp = nn.Sequential(
                nn.Linear(96, 96), nn.GELU(), nn.Linear(96, 64)
            )
            if kind == "sasrec":
                self.positions = nn.Embedding(seq_len, 64)
                layer = nn.TransformerEncoderLayer(
                    64, 2, 128, 0.15, batch_first=True, norm_first=True
                )
                self.transformer = nn.TransformerEncoder(
                    layer, 2, enable_nested_tensor=False
                )
                self.norm = nn.LayerNorm(64)

        def item_vectors(self, tokens):
            if self.kind == "sasrec":
                return F.normalize(self.output(tokens), dim=-1)
            values = torch.cat(
                [self.output(tokens), self.artist(self.artists[tokens])], dim=-1
            )
            return F.normalize(self.item_mlp(values), dim=-1)

        def user_vectors(self, pair):
            tokens, actions = pair[:, 0], pair[:, 1]
            mask = tokens > 0
            length = mask.sum(dim=1).clamp(min=1)
            if self.kind == "sasrec":
                positions = torch.arange(seq_len, device=tokens.device)
                x = self.input(tokens) + self.positions(positions)[None]
                causal = torch.ones(
                    (seq_len, seq_len), dtype=torch.bool, device=tokens.device
                ).triu(1)
                # Inputs are right-padded; the first position always attends to itself.
                x = self.transformer(x, mask=causal)
                result = self.norm(
                    x[torch.arange(len(tokens), device=tokens.device), length - 1]
                )
            else:
                values = torch.cat(
                    [
                        self.input(tokens),
                        self.artist(self.artists[tokens]),
                        self.action(actions),
                    ],
                    dim=-1,
                )
                position = torch.arange(seq_len, device=tokens.device)[None]
                weight = (
                    torch.exp2(-(length[:, None] - 1 - position).clamp(min=0) / 10)
                    * mask
                )
                result = self.user_mlp(
                    (values * weight[:, :, None]).sum(1)
                    / weight.sum(1, keepdim=True).clamp(min=1)
                )
            return F.normalize(result, dim=-1)

    for name in names:
        if name == "multvae":
            vae_source(state, cols, device)
            continue
        torch.manual_seed(cfg.seed)
        rng = np.random.default_rng(cfg.seed)
        start = time.monotonic()
        model = Retriever(name).to(device)
        checkpoint = state.folder / f"{name}.pt"
        training_log = []
        if checkpoint.exists():
            model.load_state_dict(
                torch.load(checkpoint, map_location=device, weights_only=True)
            )
        else:
            training, validation = [], []
            for uid, (tokens, times) in sequences.items():
                if len(tokens) < 3:
                    continue
                # Last event is held out inside past-only data; never an outer-fold target.
                validation.append((uid, len(tokens) - 1))
                training.extend(
                    (uid, p) for p in range(max(1, len(tokens) - 7), len(tokens) - 1)
                )
            if not training:
                raise ValueError("No neural chronological training examples")
            if len(training) > cfg.neural_examples:
                training = [
                    training[i]
                    for i in rng.choice(
                        len(training), cfg.neural_examples, replace=False
                    )
                ]
            validation = [
                validation[i]
                for i in rng.choice(
                    len(validation), min(256, len(validation)), replace=False
                )
            ]
            optimizer = torch.optim.AdamW(
                model.parameters(), lr=0.001, weight_decay=1e-5
            )
            best, stale = -np.inf, 0
            for epoch in range(cfg.neural_epochs):
                model.train()
                total, steps = 0.0, 0
                order = rng.permutation(len(training))
                for left in range(0, len(order), cfg.neural_batch):
                    examples = [
                        training[i] for i in order[left : left + cfg.neural_batch]
                    ]
                    x = np.stack(
                        [
                            prefixes(u, *sequences[u], p, name == "two_tower")
                            for u, p in examples
                        ]
                    )
                    targets = np.asarray([sequences[u][0][p] for u, p in examples])
                    x = torch.as_tensor(x, device=device)
                    positive = torch.as_tensor(targets, device=device)
                    # In-batch positives plus shared uniform negatives. Mask duplicate positives.
                    negative = torch.randint(
                        1, n + 1, (cfg.negative_items,), device=device
                    )
                    candidates = torch.cat([positive, negative])
                    logits = (
                        model.user_vectors(x) @ model.item_vectors(candidates).T / 0.1
                    )
                    duplicates = positive[:, None] == candidates[None]
                    duplicates[
                        torch.arange(len(examples), device=device),
                        torch.arange(len(examples), device=device),
                    ] = False
                    logits = logits.masked_fill(duplicates, -1e4)
                    loss = F.cross_entropy(
                        logits, torch.arange(len(examples), device=device)
                    )
                    if not torch.isfinite(loss):
                        raise ValueError(f"Non-finite {name} loss")
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 5)
                    optimizer.step()
                    total += float(loss.detach())
                    steps += 1
                    if time.monotonic() - start > cfg.neural_minutes * 60:
                        break
                model.eval()
                with torch.no_grad():
                    vx = np.stack(
                        [
                            prefixes(u, *sequences[u], p, name == "two_tower")
                            for u, p in validation
                        ]
                    )
                    target = np.asarray([sequences[u][0][p] - 1 for u, p in validation])
                    iv = (
                        model.item_vectors(torch.arange(1, n + 1, device=device))
                        .cpu()
                        .numpy()
                    )
                    uv = (
                        model.user_vectors(torch.as_tensor(vx, device=device))
                        .cpu()
                        .numpy()
                    )
                    hits = []
                    for left in range(0, len(validation), 32):
                        scores = uv[left : left + 32] @ iv.T
                        top = np.argpartition(scores, -min(100, n), axis=1)[
                            :, -min(100, n) :
                        ]
                        hits.extend(
                            np.any(top == target[left : left + 32, None], axis=1)
                        )
                    proxy = float(np.mean(hits))
                training_log.append(
                    dict(
                        epoch=epoch + 1,
                        steps=steps,
                        loss=total / max(steps, 1),
                        inner_hit100=proxy,
                        seconds=time.monotonic() - start,
                    )
                )
                print(name, training_log[-1], flush=True)
                if proxy > best:
                    best, stale = proxy, 0
                    tmp = checkpoint.with_suffix(".partial.pt")
                    torch.save(model.state_dict(), tmp)
                    tmp.replace(checkpoint.with_suffix(".best.pt"))
                else:
                    stale += 1
                if stale >= 2 or time.monotonic() - start > cfg.neural_minutes * 60:
                    break
            pd.DataFrame(training_log).to_csv(
                state.folder / f"{name}_learning.csv", index=False
            )
            checkpoint.with_suffix(".best.pt").replace(checkpoint)
            model.load_state_dict(
                torch.load(checkpoint, map_location=device, weights_only=True)
            )
        model.eval()
        with torch.no_grad():
            vectors = (
                model.item_vectors(torch.arange(1, n + 1, device=device)).cpu().numpy()
            )
            user_vectors = []
            for left in range(0, len(state.users), 256):
                examples = []
                for uid in state.users[left : left + 256]:
                    tokens, times = sequences.get(
                        int(uid), (np.array([], np.int64), np.array([], np.int64))
                    )
                    examples.append(
                        prefixes(
                            int(uid), tokens, times, len(tokens), name == "two_tower"
                        )
                    )
                user_vectors.append(
                    model.user_vectors(
                        torch.as_tensor(np.stack(examples), device=device)
                    )
                    .cpu()
                    .numpy()
                )
            user_vectors = np.concatenate(user_vectors)
        bank = state.dense_bank(user_vectors, vectors, items)
        # Empty profiles should not turn into random embedding recommendations.
        for uid in state.users:
            if int(uid) not in sequences and (
                name != "two_tower" or int(uid) not in heard
            ):
                bank[int(uid)] = ([], [])
        state.store_bank(name, bank, items, time.monotonic() - start)
        # Check ANN separately from model quality; exact retrieval is cheap at 50k items.
        if name == "two_tower":
            import faiss

            faiss.omp_set_num_threads(1)
            index = faiss.IndexHNSWFlat(64, 32, faiss.METRIC_INNER_PRODUCT)
            index.hnsw.efConstruction = 80
            index.hnsw.efSearch = max(128, cfg.source_k)
            index.add(np.asarray(vectors, np.float32))
            _, idx = index.search(
                np.asarray(user_vectors[:100], np.float32), min(cfg.source_k, n)
            )
            overlap = []
            for row, uid in enumerate(state.users[:100]):
                exact = set(bank[int(uid)][0])
                if exact:
                    overlap.append(len(exact & set(items[idx[row]])) / len(exact))
            write_json(
                state.folder / "two_tower_ann.json",
                {
                    "ann_exact_overlap": float(np.mean(overlap)) if overlap else None,
                    "queries": len(overlap),
                    "production_retrieval": "exact",
                },
            )
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
def vae_source(state, cols, device):
    import torch
    from torch import nn
    from torch.nn import functional as F

    cfg = state.cfg
    torch.manual_seed(cfg.seed)
    rng = np.random.default_rng(cfg.seed)
    n = len(cols)
    matrix = state.B[:, cols].tocsr()
    rows = np.flatnonzero(np.diff(matrix.indptr) > 1)
    if len(rows) > 20000:
        rows = rng.choice(rows, 20000, replace=False)

    class VAE(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = nn.Sequential(
                nn.Linear(n, 128), nn.Tanh(), nn.Linear(128, 128)
            )
            self.decoder = nn.Sequential(
                nn.Linear(64, 128), nn.Tanh(), nn.Linear(128, n)
            )

        def forward(self, x):
            mu, logvar = self.encoder(
                F.dropout(F.normalize(x, p=2, dim=1), 0.5, self.training)
            ).chunk(2, dim=1)
            logvar = logvar.clamp(-10, 10)
            z = (
                mu + torch.randn_like(mu) * torch.exp(0.5 * logvar)
                if self.training
                else mu
            )
            return self.decoder(z), mu, logvar

    model = VAE().to(device)
    path = state.folder / "multvae.pt"
    start = time.monotonic()
    if path.exists():
        model.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    else:
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        step = 0
        for epoch in range(min(cfg.neural_epochs, 5)):
            model.train()
            for left in range(0, len(rows), 128):
                if left == 0:
                    rng.shuffle(rows)
                x = torch.as_tensor(
                    matrix[rows[left : left + 128]].toarray(), device=device
                )
                logits, mu, logvar = model(x)
                recon = -(F.log_softmax(logits, dim=1) * x).sum(1).mean()
                kl = -0.5 * (1 + logvar - mu.square() - logvar.exp()).sum(1).mean()
                loss = recon + min(0.2, step / 1000) * kl
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
                step += 1
                if time.monotonic() - start > cfg.neural_minutes * 60:
                    break
            if time.monotonic() - start > cfg.neural_minutes * 60:
                break
        temporary = path.with_suffix(".partial.pt")
        torch.save(model.state_dict(), temporary)
        temporary.replace(path)
    model.eval()
    bank = {}
    with torch.no_grad():
        for left in range(0, len(state.users), 32):
            x = torch.as_tensor(
                matrix[state.user_rows[left : left + 32]].toarray(), device=device
            )
            logits, _, _ = model(x)
            for row, values in enumerate(logits.cpu().numpy(), left):
                bank[int(state.users[row])] = best_k(
                    state.items[cols], values, cfg.source_k
                )
    state.store_bank("multvae", bank, state.items[cols], time.monotonic() - start)

In [ ]:
def bpr_source(state, requested):
    if "bpr" not in requested or state.cached_bank("bpr"):
        return
    from implicit.bpr import BayesianPersonalizedRanking

    start = time.monotonic()
    model = BayesianPersonalizedRanking(
        factors=state.cfg.als_factors,
        iterations=20,
        regularization=0.01,
        learning_rate=0.03,
        random_state=state.cfg.seed,
        num_threads=state.cfg.threads,
        use_gpu=False,
    )
    with threadpool_limits(limits=1, user_api="blas"):
        model.fit(state.B, show_progress=False)
    bank = state.dense_bank(
        model.user_factors[state.user_rows], model.item_factors, state.items
    )
    for row, uid in enumerate(state.users):
        if state.B[state.user_rows[row]].nnz == 0:
            bank[int(uid)] = (np.array([], np.int64), np.array([], np.float32))
    state.store_bank("bpr", bank, state.items, time.monotonic() - start)

In [ ]:
def build_sources(state, requested):
    requested = list(requested)
    for label, fn in [
        ("popularity and metadata", simple_sources),
        ("ALS confidence and retrieval", als_sources),
        ("neighborhoods", neighborhood_sources),
        ("temporal transitions", transition_sources),
        ("linear autoencoders", linear_sources),
        ("neural retrievers", neural_sources),
        ("BPR", bpr_source),
    ]:
        with state.e.stage(f"{label} cut={state.cut}"):
            fn(state, requested)
    missing = set(requested) - set(state.banks)
    if missing:
        raise ValueError(f"Requested sources were not built: {sorted(missing)}")
    return state

## Отбор источников и предел покрытия

Для каждого источника выводятся catalog ceiling, Recall@300 и Recall@100. Каталоги EASE/SANSA/нейросетей различаются, поэтому низкий Recall рассматривается вместе с долей доступных целевых треков. Финальный пул ограничен 1200 кандидатами: небольшая квота сохраняет вклад каждого источника, затем свободные места заполняются reciprocal rank fusion.

Greedy-отбор добавляет источник с максимальным приростом покрытия **при том же итоговом бюджете**. Это быстрая эвристика, не исчерпывающий поиск ансамбля. Параметры источников и shortlist фиксируются до открытия позднего теста.

In [ ]:
def combine(state, names, k=None):
    k = state.cfg.candidate_k if k is None else k
    bank = {}
    for uid in state.users:
        scores = {}
        reserved = set()
        for name in names:
            ids, _ = state.banks[name][int(uid)]
            reserved.update(map(int, ids[: min(40, k // max(1, len(names)))]))
            for rank, item in enumerate(ids):
                scores[int(item)] = scores.get(int(item), 0.0) + 1 / (60 + rank + 1)
        # Reserve a small quota per source, then fill by reciprocal rank fusion.
        ordered = sorted(scores, key=lambda i: (-scores[i], i))
        chosen = sorted(reserved, key=lambda i: (-scores[i], i))[:k]
        chosen += [i for i in ordered if i not in reserved][: k - len(chosen)]
        chosen = sorted(chosen, key=lambda i: (-scores[i], i))
        bank[int(uid)] = (
            np.asarray(chosen, np.int64),
            np.asarray([scores[i] for i in chosen], np.float32),
        )
    return bank

In [ ]:
def audit_sources(state, truth, counts):
    rows = []
    for name, bank in state.banks.items():
        top100, _ = measure(bank, truth, counts, 100)
        recall, _ = measure(bank, truth, counts, state.cfg.source_k)
        catalog = set(map(int, state.catalogs[name]))
        catalog_recalls = [
            len(target & catalog) / len(target)
            for u, target in truth.items()
            if counts[u] >= 100 and target
        ]
        rows.append(
            dict(
                source=name,
                recall100=top100["full100"],
                active100=top100["active"],
                recall_candidates=recall["full100"],
                catalog_ceiling=np.mean(catalog_recalls) if catalog_recalls else np.nan,
                catalog=len(catalog),
                seconds=state.source_seconds[name],
                mean_candidates=np.mean([len(v[0]) for v in bank.values()]),
                users_with_candidates=float(
                    np.mean([bool(len(v[0])) for v in bank.values()])
                ),
            )
        )
    frame = pd.DataFrame(rows).sort_values("recall_candidates", ascending=False)
    state.e.table("dev_source_audit", frame)
    ax = frame.set_index("source")[
        ["catalog_ceiling", "recall_candidates", "recall100"]
    ].plot.barh(figsize=(10, 8))
    ax.set_xlabel("Recall, complete next-100 cohort")
    ax.figure.tight_layout()
    ax.figure.savefig(state.e.out / "candidate_attrition.svg")
    plt.show()
    return frame

In [ ]:
def select_sources(state, truth, counts):
    if (
        sum(v >= 100 for v in counts.values()) < state.cfg.minimum_complete
        and state.cfg.primary == "full100"
    ):
        raise ValueError(
            "Too few complete users to select reliably. Increase the cohort in a new RUN_LABEL; do not silently change the metric."
        )
    selected, trace = [], []
    remaining = sorted(state.banks)
    previous = 0.0
    for step in range(min(state.cfg.max_sources, len(remaining))):
        choices = []
        for name in remaining:
            metrics, _ = measure(
                combine(state, selected + [name]), truth, counts, state.cfg.candidate_k
            )
            choices.append((metrics[state.cfg.primary], name))
        score, name = max(choices, key=lambda row: (row[0], row[1]))
        if selected and score <= previous + 1e-5:
            break
        selected.append(name)
        remaining.remove(name)
        trace.append(
            dict(
                step=step + 1,
                added=name,
                recall_candidates=score,
                gain=score - previous,
            )
        )
        previous = score
    state.e.table("source_selection", pd.DataFrame(trace))
    write_json(state.e.out / "selected_sources.json", selected)
    return selected

## Признаки и обучение ranker

Признаки описывают популярность 2/7/30 дней, скорость тренда, organic share, прошлые лайки, квалифицированные/полные/короткие прослушивания, обратную связь, предпочтения артистов и альбомов, их текущие тренды и положение в списках кандидатов. Предполагаемый срок до 100 лайков оценивается по прошлой активности пользователя; взаимодействие этого срока с краткосрочным трендом — проверяемая гипотеза, не известная закономерность.

Для ALS считаются dot product, cosine similarity и нормы на **всём объединённом пуле**, включая кандидатов других источников. Объединение не обогащается будущими позитивами. Для обучения сохраняются все найденные позитивы, 50 верхних негативов и случайная часть остальных; оценка использует весь пул и полный знаменатель целевого множества.

Logloss, QuerySoftMax и YetiRank сравниваются на одном sampled pool без весов объектов. Вероятности сэмплирования сохраняются, но это не propensities показов и не контрфактическая оценка. Неполное будущее остаётся источником false negatives. Bootstrap-интервалы на development описывают сравнения после отбора, а не подтверждают статистическую значимость всего перебора; поздний тест оценивается отдельно. Один дополнительный refit с сокращённым набором признаков проверяет pruning; importance сама по себе не считается доказательством бесполезности признака. Ещё один контроль обучает победивший вариант только на полностью наблюдаемых пользователях, если таких пользователей достаточно: он проверяет влияние цензурирования и отбирается на том же development-окне. Итого максимум пять обучений для выбора ranker, затем один refit выбранной конфигурации перед поздним тестом. Это не отдельный fit для каждой длины ансамбля.

In [ ]:
def feature_frame(state, uid, ids, fusion_scores, names):
    c = state.cut
    ids = np.asarray(ids, np.int64)
    stats = state.stats.reindex(ids).fillna(0)
    x = pd.DataFrame(index=np.arange(len(ids)))
    for name in ["pop", "pop30", "pop7", "pop2", "decay30", "decay7"]:
        x["log_" + name] = np.log1p(stats[name].to_numpy())
    x["trend7_30"] = (stats.pop7.to_numpy() + 1) / (stats.pop30.to_numpy() / 30 * 7 + 1)
    x["trend2_7"] = (stats.pop2.to_numpy() + 1) / (stats.pop7.to_numpy() / 7 * 2 + 1)
    x["item_organic_share"] = stats.organic_share.to_numpy()
    x["item_age_days"] = np.clip(
        (c - stats.first_time.to_numpy(dtype=float)) / DAY, 0, 1000
    )
    x["fusion_score"] = fusion_scores
    x["fusion_rr"] = 1 / (1 + np.arange(len(ids)))
    history = state.like_by_user.get(uid, pd.DataFrame(columns=state.like.columns))
    heard = state.listen_by_user.get(uid, pd.DataFrame(columns=state.listen.columns))
    x["user_like_history"] = np.log1p(len(history))
    x["user_listen_history"] = np.log1p(len(heard))
    user30 = int((history.timestamp >= c - 30 * DAY).sum())
    horizon = min(180, 100 * 30 / max(1, user30))
    x["user_likes30"] = np.log1p(user30)
    x["estimated_100_like_days"] = horizon
    x["personal_trend"] = x["trend7_30"] * min(1, 7 / max(1, horizon))
    liked = history.set_index("item_id").reindex(ids)
    x["liked_before"] = liked.timestamp.notna().to_numpy(dtype=float)
    x["days_since_like"] = np.clip(
        (c - liked.timestamp.fillna(c - 1000 * DAY).to_numpy(dtype=float)) / DAY,
        0,
        1000,
    )
    listening = heard.set_index("item_id").reindex(ids)
    for name in ["n", "qualified", "full", "short", "recent7"]:
        x["listen_" + name] = np.log1p(listening[name].fillna(0).to_numpy())
    x["listen_organic"] = listening.organic.fillna(0).to_numpy()
    x["days_since_listen"] = np.clip(
        (c - listening.timestamp.fillna(c - 1000 * DAY).to_numpy(dtype=float)) / DAY,
        0,
        1000,
    )
    feedback = state.feedback_by_user.get(
        uid, pd.DataFrame(columns=["ever_disliked", "ever_unliked"])
    )
    for name in ["ever_disliked", "ever_unliked"]:
        x[name] = feedback.reindex(ids)[name].fillna(0).to_numpy()
    for kind in ["artist", "album"]:
        mapping = getattr(state, kind)
        groups = mapping.reindex(ids).fillna(-1).to_numpy()
        past = mapping.reindex(history.item_id).fillna(-1).to_numpy()
        all_counts = pd.Series(past[past >= 0]).value_counts()
        recent = past[history.timestamp.to_numpy(dtype=float) >= c - 30 * DAY]
        recent_counts = pd.Series(recent[recent >= 0]).value_counts()
        x[f"{kind}_likes"] = np.log1p(all_counts.reindex(groups).fillna(0).to_numpy())
        x[f"{kind}_likes30"] = np.log1p(
            recent_counts.reindex(groups).fillna(0).to_numpy()
        )
        # Group trend is computed from past-only global track statistics.
        if not hasattr(state, "_group_trends"):
            state._group_trends = {}
        if kind not in state._group_trends:
            grouped = (
                state.stats[["pop30", "pop7"]]
                .join(mapping.rename("group"))
                .dropna()
                .groupby("group")
                .sum()
            )
            state._group_trends[kind] = (grouped.pop7 + 1) / (
                grouped.pop30 * 7 / 30 + 1
            )
        x[f"{kind}_trend"] = (
            state._group_trends[kind].reindex(groups).fillna(1).to_numpy()
        )
    for name in names:
        source_ids, scores = state.banks[name][uid]
        lookup = {
            int(item): (float(score), rank + 1)
            for rank, (item, score) in enumerate(zip(source_ids, scores))
        }
        values = [lookup.get(int(item), (0.0, 0)) for item in ids]
        x[name + "_score"] = [v[0] for v in values]
        x[name + "_rr"] = [1 / v[1] if v[1] else 0 for v in values]
    for name in sorted({n.removesuffix("_cos") for n in names if n.startswith("als_")}):
        if name not in state.factor_features:
            saved = np.load(state.folder / f"{name}_model.npz")
            state.factor_features[name] = (saved["users"], saved["items"])
        uf, vf = state.factor_features[name]
        user = uf[state.query_rows[uid]]
        positions = np.searchsorted(state.items, ids)
        valid = positions < len(state.items)
        valid[valid] &= state.items[positions[valid]] == ids[valid]
        vectors = np.zeros((len(ids), vf.shape[1]), np.float32)
        vectors[valid] = vf[positions[valid]]
        dot = vectors @ user
        user_norm = np.linalg.norm(user)
        item_norm = np.linalg.norm(vectors, axis=1)
        x[name + "_dot_all"] = dot
        x[name + "_cos_all"] = dot / np.maximum(user_norm * item_norm, 1e-12)
        x[name + "_item_norm"] = item_norm
        x[name + "_user_norm"] = user_norm
    matrix = x.to_numpy(dtype=np.float32)
    if not np.isfinite(matrix).all():
        row, col = np.argwhere(~np.isfinite(matrix))[0]
        raise ValueError(
            f"Nonfinite uid={uid}, item={ids[row]}, feature={x.columns[col]}"
        )
    return x.astype(np.float32)

In [ ]:
def write_pairs(state, names, truth, counts, split, training):
    key = fingerprint(dict(users=state.users.tolist(), names=names, training=training))
    path = state.e.cache / f"pairs_{split}_{key}.parquet"
    meta = path.with_suffix(".json")
    if path.exists() and meta.exists():
        return path, json.loads(meta.read_text())
    import pyarrow as pa
    import pyarrow.parquet as pq

    bank = combine(state, names)
    writer = None
    rows = 0
    feature_names = None
    tmp = path.with_suffix(".partial.parquet")
    rng = np.random.default_rng(state.cfg.seed)
    try:
        for user_index, (uid, (ids, scores)) in enumerate(bank.items()):
            if user_index % 250 == 0:
                print(
                    f"PAIRS {split}: {user_index}/{len(bank)} users; {rows:,} rows",
                    flush=True,
                )
            if len(ids) == 0:
                continue
            labels = np.isin(ids, list(truth[uid])).astype(np.int8)
            probability = np.ones(len(ids), np.float32)
            keep = np.arange(len(ids))
            if training:
                if not labels.any():
                    continue
                positive = np.flatnonzero(labels)
                negative = np.flatnonzero(labels == 0)
                hard = negative[: min(50, state.cfg.negative_rows)]
                rest = negative[len(hard) :]
                sample_n = min(len(rest), max(0, state.cfg.negative_rows - len(hard)))
                random = rng.choice(rest, sample_n, replace=False)
                if len(rest):
                    probability[rest] = sample_n / len(rest)
                keep = np.sort(np.r_[positive, hard, random])
            features = feature_frame(state, uid, ids[keep], scores[keep], names)
            # Recompute the original fusion rank after negative sampling.
            features["fusion_rr"] = 1 / (1 + keep)
            feature_names = list(features)
            features.insert(0, "uid", np.int64(uid))
            features.insert(1, "item_id", ids[keep])
            features["label"] = labels[keep]
            features["weight"] = 1 / np.maximum(probability[keep], 1e-12) / len(ids)
            features["full100"] = counts[uid] >= 100
            table = pa.Table.from_pandas(features, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(tmp, table.schema, compression="zstd")
            writer.write_table(table, row_group_size=len(table))
            rows += len(table)
    finally:
        if writer is not None:
            writer.close()
    if not rows:
        raise ValueError("No candidate rows with positive training labels")
    tmp.replace(path)
    receipt = {
        "features": feature_names,
        "rows": rows,
        "sources": names,
        "training": training,
    }
    write_json(meta, receipt)
    return path, receipt

In [ ]:
def merge_pair_windows(e, paths, tag):
    import pyarrow as pa

    signature = [
        (str(Path(p).resolve()), Path(p).stat().st_size, Path(p).stat().st_mtime_ns)
        for p in paths
    ]
    destination = e.cache / f"{tag}_{fingerprint(signature)}.parquet"
    if destination.exists():
        return destination
    temporary = destination.with_suffix(".partial.parquet")
    writer = None
    try:
        for fold, path in enumerate(paths):
            for batch in pq.ParquetFile(path).iter_batches(batch_size=65536):
                frame = batch.to_pandas()
                frame.uid = frame.uid.astype(np.int64) + (fold << 40)
                table = pa.Table.from_pandas(frame, preserve_index=False)
                if writer is None:
                    writer = pq.ParquetWriter(
                        temporary, table.schema, compression="zstd"
                    )
                writer.write_table(table)
    finally:
        if writer is not None:
            writer.close()
    temporary.replace(destination)
    return destination

In [ ]:
def train_ranker(e, pairs, features, loss, tag, complete_only=False, iterations=None):
    from catboost import CatBoostClassifier, CatBoostRanker, Pool

    frame = pd.read_parquet(pairs).sort_values("uid", kind="stable")
    if complete_only:
        frame = frame[frame.full100]
    iterations = iterations or max(e.cfg.trees)
    pair_stat = Path(pairs).stat()
    model_key = fingerprint(
        {
            "features": features,
            "pairs": str(Path(pairs).resolve()),
            "size": pair_stat.st_size,
            "mtime_ns": pair_stat.st_mtime_ns,
            "complete_only": complete_only,
            "iterations": iterations,
        }
    )
    path = e.cache / f"ranker_{tag}_{model_key}.cbm"
    cls = CatBoostClassifier if loss == "Logloss" else CatBoostRanker
    model = cls(
        iterations=iterations,
        depth=6,
        learning_rate=0.05,
        l2_leaf_reg=10,
        loss_function=loss,
        task_type=e.cfg.task_type,
        thread_count=e.cfg.threads,
        random_seed=e.cfg.seed,
        allow_writing_files=False,
        verbose=False,
    )
    if path.exists():
        model.load_model(path)
        return model
    # Match the negative pool and query weighting across objectives.
    # Ranking losses use unweighted sampled groups; Logloss is also unweighted here.
    pool = Pool(
        frame[features],
        label=frame.label,
        group_id=None if loss == "Logloss" else frame.uid,
    )
    with e.stage("CatBoost " + tag):
        model.fit(pool)
    temporary = path.with_suffix(".partial.cbm")
    model.save_model(temporary, format="cbm")
    temporary.replace(path)
    return model

In [ ]:
def predict_ranker(model, pairs, features, trees):
    bank = {}
    parquet = pq.ParquetFile(pairs)
    for group in range(parquet.num_row_groups):
        frame = parquet.read_row_group(group).to_pandas()
        values = np.asarray(
            model.predict(
                frame[features],
                ntree_end=trees,
                **(
                    {"prediction_type": "RawFormulaVal"}
                    if hasattr(model, "predict_proba")
                    else {}
                ),
            )
        )
        # Row groups contain exactly one user, constructed explicitly by write_pairs.
        for uid, rows in frame.groupby("uid", sort=False):
            indices = rows.index.to_numpy()
            bank[int(uid)] = best_k(rows.item_id, values[indices], 100)
    return bank

In [ ]:
def paired_interval(first, second, truth, counts, primary, seed):
    _, a = measure(first, truth, counts)
    _, b = measure(second, truth, counts)
    mask = a.full100 if primary == "full100" else a.active
    differences = (a.recall - b.recall)[mask].to_numpy()
    rng = np.random.default_rng(seed)
    samples = (
        [
            rng.choice(differences, len(differences), replace=True).mean()
            for _ in range(1000)
        ]
        if len(differences)
        else [np.nan]
    )
    return {
        "delta": float(np.mean(differences)) if len(differences) else None,
        "ci_low": float(np.quantile(samples, 0.025)),
        "ci_high": float(np.quantile(samples, 0.975)),
        "n": len(differences),
    }

In [ ]:
def ranker_search(e, train_pairs, dev_pairs, features, truth, counts):
    results, models, predictions = [], {}, {}
    for loss in ["Logloss", "QuerySoftMax", "YetiRank"]:
        model = train_ranker(e, train_pairs, features, loss, loss)
        models[loss] = model
        for trees in e.cfg.trees:
            bank = predict_ranker(model, dev_pairs, features, trees)
            metrics, _ = measure(bank, truth, counts)
            results.append(dict(recipe=loss, trees=trees, **metrics))
            predictions[(loss, trees)] = bank
    table = pd.DataFrame(results).sort_values(e.cfg.primary, ascending=False)
    e.table("dev_rankers", table)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for metric, ax in zip(["full100", "active"], axes):
        for loss, part in table.groupby("recipe"):
            part = part.sort_values("trees")
            ax.plot(part.trees, part[metric], marker="o", label=loss)
        ax.set(xlabel="Trees", ylabel="Recall@100", title=metric)
        ax.legend()
    fig.tight_layout()
    fig.savefig(e.out / "ranker_learning_curve.svg")
    plt.show()
    best = table.iloc[0]
    winner = dict(
        loss=best.recipe,
        trees=int(best.trees),
        features=features,
        training_subset="all",
    )
    # A single pruned refit, selected on development only, replaces a large feature sweep.
    model = models[winner["loss"]]
    from catboost import Pool

    sample = pd.read_parquet(train_pairs)
    pool = Pool(
        sample[features],
        label=sample.label,
        group_id=None if winner["loss"] == "Logloss" else sample.uid,
    )
    importance = model.get_feature_importance(pool, type="PredictionValuesChange")
    importance_table = pd.DataFrame(
        {"feature": features, "importance": importance}
    ).sort_values("importance", ascending=False)
    e.table("feature_importance", importance_table)
    retained = importance_table.head(
        max(20, math.ceil(len(features) * 0.6))
    ).feature.tolist()
    pruned = train_ranker(
        e,
        train_pairs,
        retained,
        winner["loss"],
        "pruned_" + winner["loss"],
        iterations=winner["trees"],
    )
    pruned_bank = predict_ranker(pruned, dev_pairs, retained, winner["trees"])
    pruned_metrics, _ = measure(pruned_bank, truth, counts)
    reference = predictions[(winner["loss"], winner["trees"])]
    comparison = paired_interval(
        pruned_bank, reference, truth, counts, e.cfg.primary, e.cfg.seed
    )
    e.table(
        "feature_pruning",
        pd.DataFrame([{**pruned_metrics, **comparison, "features": len(retained)}]),
    )
    # Keep full features unless pruning is non-worse by the point estimate.
    if pruned_metrics[e.cfg.primary] >= float(best[e.cfg.primary]):
        winner["features"] = retained
        winner["pruned"] = True
        model = pruned
    else:
        winner["pruned"] = False
    # One direct censoring control on the winning feature set, without regenerating pairs.
    complete_users = sample.loc[sample.full100, "uid"].nunique()
    if complete_users >= e.cfg.minimum_complete:
        complete = train_ranker(
            e,
            train_pairs,
            winner["features"],
            winner["loss"],
            "complete_" + winner["loss"],
            complete_only=True,
            iterations=winner["trees"],
        )
        complete_bank = predict_ranker(
            complete, dev_pairs, winner["features"], winner["trees"]
        )
        complete_metrics, _ = measure(complete_bank, truth, counts)
        current_bank = pruned_bank if winner["pruned"] else reference
        current_metrics, _ = measure(current_bank, truth, counts)
        difference = paired_interval(
            complete_bank, current_bank, truth, counts, e.cfg.primary, e.cfg.seed
        )
        e.table(
            "censoring_control",
            pd.DataFrame(
                [
                    {
                        **complete_metrics,
                        **difference,
                        "train_complete_users": complete_users,
                    }
                ]
            ),
        )
        if complete_metrics[e.cfg.primary] > current_metrics[e.cfg.primary]:
            winner["training_subset"] = "full100"
            model = complete
    else:
        write_json(
            e.out / "censoring_control_skipped.json",
            {
                "complete_train_users": int(complete_users),
                "minimum": e.cfg.minimum_complete,
            },
        )
    write_json(e.out / "frozen_ranker.json", winner)
    return winner, model, table

## Графики, отчёт и формат посылки

HTML и ZIP содержат только результаты текущего запуска, конфигурацию, временные измерения и learning curves. Тяжёлые модели и pair tables остаются в локальном кеше сессии. Архив результатов стоит скачать до отключения Colab; без Drive потеря виртуальной машины означает потерю её кеша. Для воспроизводимости кеш не обязателен.

Экспорт посылки реализован ниже, но выключен по умолчанию. 50M и 500M — выборки пользователей; значительная часть официальных UIDs может отсутствовать. Notebook покажет покрытие и по умолчанию не выдаст такую диагностическую посылку за полноценное решение конкурса.

In [ ]:
def initial_eda(e):
    schema = []
    for path in sorted(e.data.glob("*.parquet")):
        info = pq.ParquetFile(path)
        schema.append(
            dict(
                file=path.name,
                rows=info.metadata.num_rows,
                disk_gib=path.stat().st_size / 1024**3,
                schema=str(info.schema_arrow),
            )
        )
    e.table("schemas", pd.DataFrame(schema))
    # EDA used for choices ends at the development boundary; holdout labels stay closed.
    daily = e.db.execute(f"""
        SELECT floor(timestamp::DOUBLE/{DAY})::INTEGER AS day,count(*) AS likes,
               count(DISTINCT uid) AS active_users,avg(is_organic::DOUBLE) AS organic_share
        FROM likes WHERE timestamp<{e.cuts["test"]} GROUP BY day ORDER BY day
    """).df()
    e.table("daily_likes", daily.tail(10))
    daily.to_csv(e.out / "daily_likes.csv", index=False)
    fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    axes[0].plot(daily.day, daily.likes, label="likes/day")
    axes[1].plot(daily.day, daily.organic_share, label="organic share")
    for ax in axes:
        for name in ["train", "dev"]:
            ax.axvline(e.cuts[name] / DAY, linestyle="--", alpha=0.5, label=name)
        ax.legend()
    fig.tight_layout()
    fig.savefig(e.out / "timeline.svg")
    plt.show()
    censor = []
    for split in ["train", "dev"]:
        truth, counts = e.targets(split)
        censor.append(
            dict(
                split=split,
                start=e.cuts[split],
                end=e.cuts[split] + 60 * DAY,
                users=len(counts),
                active=sum(bool(t) for t in truth.values()),
                full100=sum(v >= 100 for v in counts.values()),
                event_vs_unique_gap=sum(counts[u] - len(truth[u]) for u in truth),
            )
        )
    e.table("target_censoring", pd.DataFrame(censor))
    if e.cfg.primary == "full100" and censor[-1]["full100"] < e.cfg.minimum_complete:
        raise ValueError(
            "Too few complete development users. Increase CFG.cohort with a new RUN_LABEL before fitting any retriever."
        )
    span = (e.end - e.start) / DAY
    print(
        f"Observed span: {span:.1f} days. Cohort: {len(e.cohort)}; official overlap: {np.isin(e.cohort, e.official).sum()}"
    )
    if not 200 < span < 400:
        raise ValueError(
            "Unexpected timestamp scale: inspect the data before interpreting days"
        )

In [ ]:
def final_audit(e, state, bank, truth, counts, names):
    rows = []
    for uid, target in truth.items():
        liked = set(
            state.like_by_user.get(uid, pd.DataFrame(columns=["item_id"])).item_id
        )
        heard = set(
            state.listen_by_user.get(uid, pd.DataFrame(columns=["item_id"])).item_id
        )
        prediction = set(bank.get(uid, ([], []))[0])
        for label, subset in [
            ("previously_liked", target & liked),
            ("heard_not_liked", (target & heard) - liked),
            ("not_heard_in_window", target - liked - heard),
        ]:
            if subset:
                rows.append(
                    dict(
                        uid=uid,
                        group=label,
                        n_targets=len(subset),
                        recall=len(prediction & subset) / len(subset),
                        full100=counts[uid] >= 100,
                    )
                )
    frame = pd.DataFrame(rows)
    e.table(
        "test_target_groups",
        frame.groupby("group")
        .agg(
            users=("uid", "size"),
            targets=("n_targets", "sum"),
            conditional_recall=("recall", "mean"),
        )
        .reset_index(),
    )
    pool = combine(state, names)
    coverage, _ = measure(pool, truth, counts, e.cfg.candidate_k)
    final, per_user = measure(bank, truth, counts)
    e.table(
        "test_metrics",
        pd.DataFrame(
            [
                dict(stage="candidate_pool", **coverage),
                dict(stage="ranker_top100", **final),
            ]
        ),
    )
    per_user.to_csv(e.out / "test_per_user.csv", index=False)
    official = [u for u in truth if u in set(e.official)]
    if official:
        metric, _ = measure(bank, truth, counts, users=official)
        e.table("test_official_overlap", pd.DataFrame([metric]))
    rr = combine(state, names, 100)
    interval = paired_interval(bank, rr, truth, counts, e.cfg.primary, e.cfg.seed)
    e.table("test_vs_rrf", pd.DataFrame([interval]))
    # Known-catalog ceiling and source overlap diagnose failures without further tuning.
    catalog = set().union(*(set(state.catalogs[n]) for n in names))
    catalog_bank = {u: (np.asarray(sorted(t & catalog)), []) for u, t in truth.items()}
    known, _ = measure(catalog_bank, truth, counts, 100)
    e.table(
        "test_attrition",
        pd.DataFrame(
            [
                dict(stage="known to selected catalogs", recall=known[e.cfg.primary]),
                dict(stage="candidate union", recall=coverage[e.cfg.primary]),
                dict(stage="top100", recall=final[e.cfg.primary]),
            ]
        ),
    )
    return final

In [ ]:
def write_report(e):
    from html import escape

    parts = [
        '<!doctype html><meta charset="utf-8"><title>Yambda v10 experiments</title>',
        "<style>body{max-width:1150px;margin:40px auto;font:15px system-ui;color:#17212b}table{border-collapse:collapse;font-size:13px}td,th{padding:6px;border:1px solid #ddd}img{max-width:100%}pre{white-space:pre-wrap}</style>",
        "<h1>Yambda: retrieval → ranking → chronological test</h1>",
        "<p>All scores below are computed by this run. The test window is not used to choose sources, features, loss or tree count.</p>",
    ]
    for path in sorted(e.out.glob("*.csv")):
        if path.name in ["cohort.csv", "daily_likes.csv", "test_per_user.csv"]:
            continue
        parts += [
            f"<h2>{escape(path.stem)}</h2>",
            pd.read_csv(path).head(100).to_html(index=False, escape=True),
        ]
    for path in sorted(e.out.glob("*.svg")):
        parts += [f"<h2>{escape(path.stem)}</h2>", path.read_text()]
    parts += [
        "<h2>Configuration</h2>",
        "<pre>" + escape((e.out / "configuration.json").read_text()) + "</pre>",
    ]
    report = e.out / "report.html"
    report.write_text("\n".join(parts), encoding="utf-8")
    archive = e.out / "research_v10_results.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
        for path in sorted(e.out.iterdir()):
            if path.suffix in [".csv", ".json", ".svg", ".html"] and path.is_file():
                z.write(path, path.name)
        for path in e.cache.glob("cut_*/*_learning.csv"):
            z.write(path, "learning/" + path.parent.name + "_" + path.name)
        for path in e.cache.glob("cut_*/two_tower_ann.json"):
            z.write(path, "learning/" + path.parent.name + "_" + path.name)
    print("REPORT:", report)
    print("RESULTS:", archive)
    return archive

In [ ]:
def submission(e, names, winner, model, allow_partial=False):
    if not len(e.official):
        raise ValueError("Official test_users.csv is required")
    population = e.db.execute(
        "SELECT DISTINCT uid::BIGINT AS uid FROM likes"
    ).fetchnumpy()["uid"]
    known = np.isin(e.official, population)
    print(
        f"Official users with like history in this scale: {known.sum()}/{len(known)} ({known.mean():.1%})"
    )
    if known.mean() < 0.95 and not allow_partial:
        raise ValueError(
            "50M/500M lacks history for many official UIDs. Submission is blocked by default; use larger data or explicitly permit a diagnostic partial-history submission."
        )
    folder = e.out / "submission"
    folder.mkdir(exist_ok=True)
    # The recipe is frozen. All three labeled windows may now train the production model.
    import pyarrow as pa

    path = e.cache / "production_train.parquet"
    if not path.exists():
        temporary = path.with_suffix(".partial.parquet")
        writer = None
        try:
            for fold, split in enumerate(["train", "dev", "test"]):
                state = State(e, e.cuts[split], e.cohort)
                build_sources(state, names)
                truth, counts = e.targets(split)
                fold_path, _ = write_pairs(state, names, truth, counts, split, True)
                for batch in pq.ParquetFile(fold_path).iter_batches(batch_size=65536):
                    frame = batch.to_pandas()
                    frame.uid = frame.uid.astype(np.int64) + (fold << 40)
                    table = pa.Table.from_pandas(frame, preserve_index=False)
                    if writer is None:
                        writer = pq.ParquetWriter(
                            temporary, table.schema, compression="zstd"
                        )
                    writer.write_table(table)
                del state
                gc.collect()
        finally:
            if writer is not None:
                writer.close()
        temporary.replace(path)
    final_model = train_ranker(
        e,
        path,
        winner["features"],
        winner["loss"],
        "production_" + winner["loss"],
        complete_only=winner.get("training_subset") == "full100",
        iterations=winner["trees"],
    )
    state = State(e, e.end, e.official)
    build_sources(state, names)
    empty = {int(u): set() for u in e.official}
    counts = dict.fromkeys(map(int, e.official), 0)
    # Pair writing and scoring stream one complete user at a time.
    pairs, meta = write_pairs(state, names, empty, counts, "final", False)
    scored = predict_ranker(final_model, pairs, winner["features"], winner["trees"])
    rrf = combine(state, names)
    fallback = state.stats.sort_values("decay30", ascending=False).index.to_list()
    variants = [[], [], []]
    for uid in e.official:
        ranked = list(map(int, scored.get(int(uid), ([], []))[0]))
        fused = list(map(int, rrf[int(uid)][0]))
        blend = {}
        for weight, sequence in [(1.0, ranked), (0.25, fused)]:
            for rank, item in enumerate(sequence):
                blend[item] = blend.get(item, 0) + weight / (60 + rank + 1)
        mixed = sorted(blend, key=lambda i: (-blend[i], i))
        for index, sequence in enumerate([ranked, mixed, fused]):
            unique = list(dict.fromkeys(sequence + fallback))[:100]
            if len(unique) != 100:
                raise ValueError("Fewer than 100 available unique items")
            variants[index].append(
                {"uid": int(uid), "item_ids": " ".join(map(str, unique))}
            )
    paths = []
    for name, rows in zip(["ranker", "ranker_rrf_blend", "rrf"], variants):
        frame = pd.DataFrame(rows)
        assert frame.uid.to_list() == e.official.tolist()
        assert all(len(x.split()) == len(set(x.split())) == 100 for x in frame.item_ids)
        path = folder / f"{name}.csv"
        frame.to_csv(path, index=False)
        paths.append(path)
    archive = e.out / "submission_three_files.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
        for path in paths:
            z.write(path, path.name)
    print("SUBMISSION:", archive)
    return archive

In [ ]:
initial_eda(E)

## 1. Широкий поиск на development-срезе

Оцениваем новые источники только здесь. Не обучаем CatBoost для каждого варианта источника: сначала выясняем, добавляет ли он будущие лайки в ограниченный пул. После этапа сохраняется предварительный отчёт, который уже можно смотреть и скачивать.

In [ ]:
SOURCES = [
    "decay7",
    "decay30",
    "favorite_artist",
    "favorite_album",
    "heard_unliked",
    "als_likes",
    "als_likes_cos",
    "als_strong",
    "als_strong_cos",
    "als_bm25",
    "als_bm25_cos",
    "als_mixed",
    "als_mixed_cos",
    "item_cosine",
    "user_knn",
    "like_to_like",
    "listen_to_like",
    "ease",
]
if RUN_SANSA:
    SOURCES.append("sansa")
if RUN_NEURAL:
    SOURCES += ["two_tower", "sasrec"]
if RUN_OPTIONAL:
    SOURCES.append("bpr")
    if RUN_NEURAL:
        SOURCES.append("multvae")

promotion = None
if PROMOTION_PATH:
    promotion = json.loads(Path(PROMOTION_PATH).read_text())
    if promotion["implementation"] != IMPLEMENTATION:
        raise ValueError("Promotion belongs to another implementation")
    SOURCES = promotion["sources"]
    if ("sansa" in SOURCES and not RUN_SANSA) or (
        set(SOURCES) & {"two_tower", "sasrec", "multvae"} and not RUN_NEURAL
    ):
        raise ValueError("Enable the dependencies required by the promoted sources")
    write_json(E.out / "input_promotion.json", promotion)

E.bind_plan(SOURCES, promotion)
DEV_TRUTH, DEV_COUNTS = E.targets("dev")
DEV = State(E, E.cuts["dev"], E.cohort)
build_sources(DEV, SOURCES)
SOURCE_AUDIT = audit_sources(DEV, DEV_TRUTH, DEV_COUNTS)
SELECTED = select_sources(DEV, DEV_TRUTH, DEV_COUNTS)

budget_rows = []
for budget in [300, 600, 1200]:
    metrics, _ = measure(combine(DEV, SELECTED, budget), DEV_TRUTH, DEV_COUNTS, budget)
    budget_rows.append({"budget": budget, **metrics})
E.table("candidate_budget_curve", pd.DataFrame(budget_rows))
write_json(
    E.out / "promotion.json",
    {
        "implementation": IMPLEMENTATION,
        "source_scale": DATA_SIZE,
        "sources": SELECTED,
        "source_configuration": dataclasses.asdict(CFG),
        "source_run_identity": E.identity,
        "note": "Source shortlist only; train and validate again at the new scale.",
    },
)
write_report(E)

## 2. Обучение и выбор ранжировщика

После отбора заново строим те же источники на более ранней истории обучения. Development-кандидаты сохраняются до освобождения памяти. Проверяем 200/400/700 деревьев одной обученной модели, затем pruned refit и контроль полного будущего; отдельного обучения каждой длины ансамбля нет.

In [ ]:
with E.stage("development feature table"):
    DEV_PAIRS, DEV_META = write_pairs(
        DEV, SELECTED, DEV_TRUTH, DEV_COUNTS, "dev", False
    )
    DEV_TRAIN_PAIRS, _ = write_pairs(DEV, SELECTED, DEV_TRUTH, DEV_COUNTS, "dev", True)
DEV_BASELINE, _ = measure(combine(DEV, SELECTED, 100), DEV_TRUTH, DEV_COUNTS)
E.table("dev_rrf_baseline", pd.DataFrame([DEV_BASELINE]))
del DEV
E.db.execute("CHECKPOINT")
gc.collect()

TRAIN_TRUTH, TRAIN_COUNTS = E.targets("train")
TRAIN = State(E, E.cuts["train"], E.cohort)
build_sources(TRAIN, SELECTED)
with E.stage("training feature table"):
    TRAIN_PAIRS, TRAIN_META = write_pairs(
        TRAIN, SELECTED, TRAIN_TRUTH, TRAIN_COUNTS, "train", True
    )
del TRAIN
gc.collect()

WINNER, MODEL, DEV_RESULTS = ranker_search(
    E, TRAIN_PAIRS, DEV_PAIRS, DEV_META["features"], DEV_TRUTH, DEV_COUNTS
)
write_report(E)
print("Frozen recipe:", WINNER)

## 3. Поздний тест зафиксированного решения

Источники заново обучаются на истории до тестового среза, но их имена и настройки уже выбраны. Ranker с уже выбранными loss, числом деревьев и признаками переобучается на train + development: эти метки полностью известны к началу теста. Параметры больше не подбираются. Это воспроизводит обновление модели перед новым периодом. После просмотра этого результата новые изменения требуют нового эксперимента; повторное использование окна для подбора превращает его в development.

Дополнительно сравниваем ранее лайкнутые, слышанные без лайка и не слышанные в окне треки. Это условные метрики разных подзадач, их нельзя складывать или напрямую выдавать за конкурсный Recall.

In [ ]:
REFIT_PAIRS = merge_pair_windows(E, [TRAIN_PAIRS, DEV_TRAIN_PAIRS], "test_refit")
MODEL = train_ranker(
    E,
    REFIT_PAIRS,
    WINNER["features"],
    WINNER["loss"],
    "test_refit_" + WINNER["loss"],
    complete_only=WINNER["training_subset"] == "full100",
    iterations=WINNER["trees"],
)
TEST_TRUTH, TEST_COUNTS = E.targets("test")
TEST = State(E, E.cuts["test"], E.cohort)
build_sources(TEST, SELECTED)
with E.stage("test feature table"):
    TEST_PAIRS, TEST_META = write_pairs(
        TEST, SELECTED, TEST_TRUTH, TEST_COUNTS, "test", False
    )
TEST_PREDICTIONS = predict_ranker(
    MODEL, TEST_PAIRS, WINNER["features"], WINNER["trees"]
)
TEST_METRICS = final_audit(E, TEST, TEST_PREDICTIONS, TEST_TRUTH, TEST_COUNTS, SELECTED)
write_json(
    E.out / "experiment_complete.json", {"identity": E.identity, "test": TEST_METRICS}
)
RESULT_ARCHIVE = write_report(E)

## 4. Подготовка трёх посылок

После завершения проверки можно включить `RUN_SUBMISSION` и выполнить эту ячейку. После фиксации результатов ranker переобучается на всех трёх размеченных окнах, retrievers — на всей доступной истории. Создаются три CSV: ranker, ranker с небольшим RRF-компонентом, чистый RRF. Это три заранее определённых варианта, а не три гарантированно равноценных решения.

В каждом CSV сохраняется исходный порядок официальных UIDs, ровно 100 уникальных item IDs через пробел и заголовок `uid,item_ids`. Архив содержит только три CSV. При недостаточном покрытии официальных пользователей нужен больший датасет; явное разрешение partial-history экспорта означает диагностическую посылку с fallback для отсутствующих пользователей.

In [ ]:
if RUN_SUBMISSION:
    del TEST
    gc.collect()
    E.started = time.monotonic()
    SUBMISSION_ARCHIVE = submission(
        E,
        SELECTED,
        WINNER,
        MODEL,
        allow_partial=ALLOW_PARTIAL_HISTORY_SUBMISSION,
    )
else:
    print("Submission is disabled. The experiment report is ready:", RESULT_ARCHIVE)

## Скачать результаты

Архив результатов небольшой: в нём нет исходных данных и model caches. Для следующего 500M-запуска достаточно `promotion.json`, если нужно ограничиться источниками, отобранными на 50M. Его отсутствие не мешает воспроизводимому полному поиску.

In [ ]:
if IN_COLAB:
    from google.colab import files

    files.download(str(RESULT_ARCHIVE))
    if RUN_SUBMISSION:
        files.download(str(SUBMISSION_ARCHIVE))
else:
    print(RESULT_ARCHIVE)

## Источники методов и пределы эксперимента

- [Yambda: описание и schema](https://huggingface.co/datasets/yandex/yambda), [статья](https://arxiv.org/abs/2505.22238). Официальная оценка на другом горизонте не является порогом качества для наших следующих 100 лайков.
- [Time to Split](https://arxiv.org/abs/2507.16289): влияние временного протокола. Здесь предсказывается статический набор, а не online next-item с обновлением истории.
- [implicit](https://github.com/benfred/implicit): ALS/BPR и BM25 input weighting.
- [SANSA, официальный код](https://github.com/glami/sansa/tree/735a782599cceb3663f8df14d75edfdf4c86874c).
- [Turning Dross Into Gold Loss](https://github.com/antklen/sasrec-bert4rec-recsys23): важность loss и количества негативов для sequential retrieval.
- [OTTO, решение 3-го места](https://github.com/TheoViel/kaggle_otto_rs): временные co-visitation signals. Наш capped transition graph — самостоятельная компактная реализация идеи, не воспроизведение решения.
- [CatBoost ranking objectives](https://catboost.ai/docs/en/concepts/loss-functions-ranking).

Это отбор направлений при ограниченных вычислениях, а не полный поиск гиперпараметров каждого семейства. Ограничения каталога, числа событий и обучения сохраняются в конфигурации и выводятся рядом с качеством. Победу на 50M нужно подтвердить на 500M: меняются поддержка треков, пользователи и разреженность. Без проверки нельзя считать уменьшенный датасет заменой 5B для финальной посылки.